# Sentiment Classification of Beauty Product Reviews

**BC3415 Individual Assignment, Option A (coding submission)**

| | |
|---|---|
| **Name** | Dylan Choo |
| **Matriculation number** | [YOUR MATRIC NO.] |
| **Dataset** | Amazon Reviews 2023, category *All_Beauty* (McAuley Lab, UC San Diego) |
| **Task** | Classify each review as **negative**, **neutral** or **positive** |

---

<a id="toc"></a>
## Table of contents

0. [Problem statement, business context and dataset](#s0)
1. [Setup and configuration](#s1)
2. [Data cleaning and preprocessing](#s2)
3. [Exploratory data analysis](#s3)
4. [Split design](#s4)
5. [One evaluation function for every model](#s5)
6. [Models M0 to M10](#s6)
7. [Leaderboard and deployment recommendation](#s7)
8. [Business insights](#s8)
9. [Reusable inference](#s9)
10. [Conclusions, limitations, ethics and future work](#s10)

[Notes for Dylan](#notes)

<a id="s0"></a>
## 0. Problem statement, business context and dataset

### The business problem
An online beauty marketplace receives thousands of product reviews every day. Product, quality and customer-service teams cannot read them all, so useful complaints (a leaking pump, a rash, a counterfeit) get buried under thousands of "love it!" reviews.

The marketplace wants a model that reads the **review text** and labels it **positive, neutral or negative**, so that:

1. **Product teams** can track sentiment per product and per brand over time.
2. **Quality and safety teams** can find negative reviews quickly, especially ones about skin reactions or fake products.
3. **The same model works where there is no star rating**: chat logs, emails, social-media posts and Q&A questions. This is the main reason to classify *text* even though reviews already have stars.
4. **Mismatches can be flagged**: a 5-star review that describes a problem ("love the colour but the pump broke") is a hidden complaint that the star rating alone would miss.

### How success is measured
* **Macro F1** (the average of the F1 scores of the three classes) is the main metric. It weights the small neutral class as much as the large positive class. Plain accuracy would reward a model that ignores neutral reviews.
* **Accuracy on a natural-mix test set** shows how the model behaves on real traffic, where most reviews are positive.
* **Training and inference time** matter for cost: the marketplace would score every new review, so a model that is 100× slower must be clearly better to justify itself.
* **Explainability** matters for trust: product teams want to see *why* a review was flagged.

### Dataset
* **Source:** Amazon Reviews 2023, released by the McAuley Lab at UC San Diego. Project page: <https://amazon-reviews-2023.github.io/>, data on Hugging Face: <https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023>.
* **Citation:** Hou, Y., Li, J., He, Z., Yan, A., Chen, X. and McAuley, J. (2024). *Bridging Language and Items for Retrieval and Recommendation.* arXiv:2403.03952.
* **Category used:** `All_Beauty` (about 700,000 reviews of about 112,000 products, 1990s to 2023). It suits the brief well because beauty reviews mix text with customer photos and contain specific business issues (scent, irritation, leakage, shade, fakes).

| Review field | Meaning |
|---|---|
| `rating` | 1 to 5 stars (used to create the label) |
| `title`, `text` | Review headline and body |
| `images` | Customer-uploaded photos (list of image URLs) |
| `asin`, `parent_asin` | Product variant ID and parent product ID |
| `user_id` | Anonymised reviewer ID |
| `timestamp` | Review time (Unix milliseconds) |
| `helpful_vote` | Number of "helpful" votes from other shoppers |
| `verified_purchase` | Whether Amazon confirmed the purchase |

The metadata file adds product-level fields (`price`, `store`, `average_rating`, `rating_number`, `main_category`), joined on `parent_asin`.

**Labels:** 1 to 2 stars = negative, 3 stars = neutral, 4 to 5 stars = positive.

<a id="s1"></a>
## 1. Setup and configuration

**Why this section exists:** every setting the notebook depends on (paths, random seed, sample sizes, which slow sections to run) lives in **one CONFIG cell**. Every trained model, data split and prediction is saved to `outputs/`, so any later section can be re-run on its own without retraining earlier models.

### 1.1 Libraries
Everything is in this one notebook. To run it on a new machine, install the libraries once (versions used for the saved run, Python 3.13):
```bash
pip install torch==2.14.1 transformers==5.19.0 huggingface_hub==2.2.0 scikit-learn==1.9.1 xgboost==3.4.1 \
            pandas==3.0.6 numpy==2.5.4 pyarrow==26.0.0 nltk==3.10.3 vaderSentiment==3.3.2 \
            matplotlib==3.11.2 seaborn==0.13.2 joblib==1.6.0 psutil==7.2.2 pillow requests jupyter
```

In [ ]:
# Standard library
import gc, html, json, os, platform, random, re, sys, time, warnings
from dataclasses import dataclass, asdict, field
from pathlib import Path

# Data and plotting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, Markdown

# Machine learning
import sklearn, joblib, scipy
import torch
import transformers

transformers.logging.set_verbosity_error()          # hide "newly initialised weights" notices
transformers.utils.logging.disable_progress_bar()
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 90, "savefig.dpi": 110, "axes.titleweight": "bold"})

print(f"Python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | "
      f"scikit-learn {sklearn.__version__} | torch {torch.__version__} | transformers {transformers.__version__}")

### 1.2 Hardware check
The deep-learning models (especially DistilBERT) are much slower without a GPU, so the sample sizes in CONFIG depend on what this machine has. The cell below detects the CPU, RAM and accelerator (NVIDIA CUDA, Apple Silicon MPS, or CPU only).

In [ ]:
def detect_device() -> str:
    """Pick the fastest available PyTorch device: CUDA GPU > Apple Silicon (MPS) > CPU."""
    if torch.cuda.is_available():
        return "cuda"
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def hardware_summary() -> pd.Series:
    cpu = platform.processor() or platform.machine()
    try:  # Linux gives a readable CPU name in /proc/cpuinfo
        cpu = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
    except Exception:
        pass
    try:
        import psutil
        ram_gb = psutil.virtual_memory().total / 1e9
    except ImportError:
        ram_gb = float("nan")
    dev = detect_device()
    gpu = torch.cuda.get_device_name(0) if dev == "cuda" else ("Apple Silicon GPU" if dev == "mps" else "none")
    return pd.Series({
        "OS": f"{platform.system()} {platform.release()}",
        "CPU": cpu,
        "CPU cores (logical)": os.cpu_count(),
        "RAM (GB)": round(ram_gb, 1),
        "PyTorch device": dev,
        "GPU": gpu,
        "Python": sys.version.split()[0],
        "Executable": sys.executable,
    }, name="value")


HARDWARE = hardware_summary()
display(HARDWARE.to_frame())

### 1.3 CONFIG: the one place to change settings

In [ ]:
@dataclass
class Config:
    # ---- Paths ---------------------------------------------------------------
    project_dir: Path = Path.cwd()
    out_dir: Path = Path.cwd() / "outputs"
    local_data_dir: Path = Path.home() / "Downloads"   # if All_Beauty.jsonl and meta_All_Beauty.jsonl are here, use them
    hf_repo: str = "McAuley-Lab/Amazon-Reviews-2023"   # otherwise download from Hugging Face
    review_file: str = "raw/review_categories/All_Beauty.jsonl"
    meta_file: str = "raw/meta_categories/meta_All_Beauty.jsonl"

    # ---- Reproducibility -----------------------------------------------------
    seed: int = 42
    n_jobs: int = max(1, (os.cpu_count() or 2))

    # ---- Split design (Section 4) --------------------------------------------
    n_per_class: int = 12_000            # cap per class for the balanced sample (also capped by the smallest class)
    split_fracs: tuple = (0.70, 0.15, 0.15)   # train / validation / test
    n_natural_test: int = 10_000         # separate test set with the real-world class mix

    # ---- Classical models ----------------------------------------------------
    n_search_iter: int = 12              # random-search candidates for the fast linear models
    n_search_iter_trees: int = 4         # fewer candidates for Random Forest / XGBoost (slow)

    # ---- Deep learning (BiLSTM) ----------------------------------------------
    lstm_vocab_size: int = 30_000
    lstm_max_len: int = 200
    lstm_epochs: int = 6
    lstm_batch: int = 64

    # ---- DistilBERT ----------------------------------------------------------
    bert_name: str = "distilbert-base-uncased"
    bert_train_n: int = 12_000           # BERT trains on a stratified subset of the SAME training split (CPU budget)
    bert_max_len: int = 128
    bert_epochs: int = 2
    bert_batch: int = 16
    bert_lr: float = 4e-5

    # ---- Flags to skip slow sections ----------------------------------------
    run_tree_models: bool = True         # M5 Random Forest, M6 XGBoost
    run_lstm: bool = True                # M7, M8
    run_bert: bool = True                # M9, M10
    run_clip_pilot: bool = True          # optional image pilot in Section 8d
    force_retrain_classical: bool = True # classical models are cheap, so retrain on every full run
    force_retrain_deep: bool = False     # deep models load from outputs/models if a checkpoint exists

    device: str = field(default_factory=detect_device)

    @property
    def dirs(self) -> dict:
        return {k: self.out_dir / k for k in ["data", "models", "predictions", "figures", "results"]}


CFG = Config()
for d in CFG.dirs.values():
    d.mkdir(parents=True, exist_ok=True)

pd.Series({k: (str(v) if isinstance(v, Path) else v) for k, v in asdict(CFG).items()}, name="value").to_frame()

### 1.4 Seeding and small helpers
`seed_everything` fixes the random seeds of Python, NumPy and PyTorch so that samples, splits and model initialisations are the same on every run.

In [ ]:
def seed_everything(seed: int = CFG.seed) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def save_json(obj, path: Path) -> None:
    Path(path).write_text(json.dumps(obj, indent=2, default=str))


def load_json(path: Path):
    return json.loads(Path(path).read_text())


def savefig(fig, name: str) -> None:
    """Save a figure to outputs/figures so it can be reused in slides or the video walkthrough."""
    fig.savefig(CFG.dirs["figures"] / f"{name}.png", bbox_inches="tight")


seed_everything()
torch.set_num_threads(CFG.n_jobs)
NOTEBOOK_T0 = time.time()
print(f"Seed {CFG.seed} set | device = {CFG.device} | torch threads = {torch.get_num_threads()} | outputs -> {CFG.out_dir}")

<a id="s2"></a>
## 2. Data cleaning and preprocessing

**Plan for this section**
1. Download the review and metadata files (cached after the first run).
2. Clean the text: remove HTML, drop empty and duplicate reviews, and combine title and body.
3. Create the three sentiment labels from the star rating.
4. Engineer extra features (lengths, photos, verified purchase, helpful votes, date, price band).
5. Build two versions of the text: an aggressive one for bag-of-words models and a light one for the neural models.

A row count is recorded after every step, so the cleaning log at the end shows exactly how many reviews each step removed.

### 2.1 Reusable text-cleaning functions
The cleaning functions are defined once, here, and reused everywhere, including by the deployed predictor in Section 9. Text at prediction time is therefore cleaned exactly the same way as the training text.

**Why keep negation and contrast words?** Standard stopword lists (including NLTK's) contain *not, no, nor, but, very, too, against* and the *n't* contractions. Removing them destroys meaning: "**not** good" becomes "good", and "nice colour **but** it broke" loses the turn that makes it a mixed (often neutral) review. The example notebook lost its negations this way. Here, negations (*not, no, never, nor, without…*), contrast words (*but, however, although, though, yet…*) and intensity words (*very, too, only, more, less…*) are **removed from the stopword list** so they survive cleaning, and contractions are expanded first (*doesn't* becomes *does not*) so the "not" is kept as its own token.

In [ ]:
# Two cleaning levels:
#   clean_light(text)     -> for BiLSTM / DistilBERT: keeps word order, punctuation and case.
#   clean_classical(text) -> for bag-of-words models: lower-case, contractions expanded, lemmatised,
#                            stopwords removed BUT negation, contrast and intensity words kept.
from functools import lru_cache

# ---------------------------------------------------------------- raw clean-up
_BR = re.compile(r"<br\s*/?>", re.I)
_TAG = re.compile(r"<[^>]+>")
_URL = re.compile(r"https?://\S+|www\.\S+")
_WS = re.compile(r"\s+")
_REPEAT = re.compile(r"(.)\1{3,}")             # "soooooo" -> "sooo"
# Amazon fills in "Five Stars" / "One Star" when a reviewer leaves the title blank.
# Keeping it would leak the label (the star rating) into the text, so it is dropped.
AUTO_TITLE = re.compile(r"^\s*(one|two|three|four|five)\s+stars?\s*[.!]*\s*$", re.I)


def strip_html(text) -> str:
    """Unescape HTML entities (&amp; -> &), turn <br /> into spaces, remove any other tags."""
    if not isinstance(text, str):
        return ""
    text = html.unescape(html.unescape(text))   # some reviews are double-escaped (&amp;#34;)
    text = _BR.sub(" ", text)
    text = _TAG.sub(" ", text)
    return _WS.sub(" ", text).strip()


def combine_title_text(title, text) -> str:
    """'Title. Body' with auto-generated star titles removed."""
    title, text = strip_html(title), strip_html(text)
    if AUTO_TITLE.match(title):
        title = ""
    if title and text:
        sep = "" if title[-1] in ".!?" else "."
        return f"{title}{sep} {text}"
    return title or text


def clean_light(text: str) -> str:
    """Light cleaning for neural models: remove URLs, squash repeated characters and whitespace."""
    text = _URL.sub(" ", text)
    text = _REPEAT.sub(r"\1\1\1", text)
    return _WS.sub(" ", text).strip()


# ------------------------------------------------------- classical (bag-of-words)
NEGATIONS = {"not", "no", "nor", "never", "none", "nothing", "neither", "nobody", "nowhere", "cannot", "without"}
CONTRAST = {"but", "however", "although", "though", "yet", "while", "whereas", "despite", "except", "unless", "otherwise"}
INTENSITY = {"very", "too", "so", "only", "more", "most", "less", "least", "few", "enough", "really", "against", "again", "just"}
KEEP_WORDS = NEGATIONS | CONTRAST | INTENSITY

_CONTRACTIONS = [
    (re.compile(r"\bwon'?t\b"), "will not"), (re.compile(r"\bcan'?t\b"), "can not"),
    (re.compile(r"\bain'?t\b"), "is not"), (re.compile(r"\bshan'?t\b"), "shall not"),
    (re.compile(r"n't\b"), " not"),
    (re.compile(r"\b(does|did|is|was|are|were|has|have|had|could|would|should|do|need|must)nt\b"), r"\1 not"),  # "doesnt"
    (re.compile(r"'re\b"), " are"), (re.compile(r"'ve\b"), " have"), (re.compile(r"'ll\b"), " will"),
    (re.compile(r"'d\b"), " would"), (re.compile(r"'m\b"), " am"), (re.compile(r"'s\b"), ""),
]
_WORD = re.compile(r"[a-z]+")

_STOPWORDS = None
_LEMMATIZER = None


def _resources():
    """Load NLTK stopwords and the WordNet lemmatiser once (downloads them if missing)."""
    global _STOPWORDS, _LEMMATIZER
    if _STOPWORDS is None:
        import nltk
        for pkg, path in [("stopwords", "corpora/stopwords"), ("wordnet", "corpora/wordnet"), ("omw-1.4", "corpora/omw-1.4")]:
            try:
                nltk.data.find(path)
            except LookupError:
                nltk.download(pkg, quiet=True)
        from nltk.corpus import stopwords
        from nltk.stem import WordNetLemmatizer
        sw = set(stopwords.words("english"))
        sw |= {"would", "could", "also"}            # frequent, carry no sentiment
        _STOPWORDS = frozenset(sw - KEEP_WORDS)     # negation / contrast / intensity words survive
        _LEMMATIZER = WordNetLemmatizer()
    return _STOPWORDS, _LEMMATIZER


@lru_cache(maxsize=300_000)
def _lemma(tok: str) -> str:
    lem = _resources()[1]
    return lem.lemmatize(lem.lemmatize(tok, "v"), "n")   # verbs first ("leaking"->"leak"), then nouns ("bottles"->"bottle")


def clean_classical(text: str) -> str:
    """Lower-case, expand contractions, keep letters only, drop stopwords (keeping KEEP_WORDS), lemmatise."""
    stop, _ = _resources()
    text = clean_light(text).lower().replace("’", "'").replace("`", "'")
    for pat, rep in _CONTRACTIONS:
        text = pat.sub(rep, text)
    toks = [t for t in _WORD.findall(text) if (len(t) > 1 or t in KEEP_WORDS) and t not in stop]
    return " ".join(_lemma(t) for t in toks)


def clean_classical_many(texts, n_jobs: int = 1, chunksize: int = 2_000) -> list:
    """clean_classical over a list, in parallel on Linux / macOS.

    Forked worker processes inherit the functions defined in this notebook, so nothing has to be pickled.
    Windows has no fork, so it falls back to a single process there (slower, same result).
    """
    import multiprocessing as mp
    texts = list(texts)
    if n_jobs <= 1 or len(texts) < 50_000 or "fork" not in mp.get_all_start_methods():
        return [clean_classical(t) for t in texts]
    _resources()   # make sure the NLTK data is loaded before forking
    with mp.get_context("fork").Pool(n_jobs) as pool:
        return pool.map(clean_classical, texts, chunksize)


# ---------------------------------------------------------------- engineered features
def text_features(texts):
    """Vectorised length / style features for a pandas Series of review texts."""
    s = pd.Series(texts).fillna("").astype(str)
    n_end = s.str.count(r"[.!?]+(?:\s|$)")                       # sentence terminators
    ends_with_end = s.str.contains(r"[.!?]\s*$")
    letters = s.str.count(r"[A-Za-z]")
    return pd.DataFrame({
        "char_count": s.str.len(),
        "word_count": s.str.count(r"\S+"),
        "sentence_count": (n_end + (~ends_with_end).astype(int)).where(s.str.len() > 0, 0),
        "exclaim_count": s.str.count("!"),
        "upper_ratio": (s.str.count(r"[A-Z]") / letters.where(letters > 0, 1)).astype(float),
    }, index=s.index)

In [ ]:
# Quick self-test on hand-written examples, so the effect of each rule is visible.
demo = pd.DataFrame({"raw": [
    "Love it!!!!!! Best mascara EVER, see https://amzn.to/abc",
    "I <b>didn't</b> like it.<br /><br />The pump broke &amp; it leaked everywhere!!!!",
    "Nice colour but it doesn't last more than 2 hours. Not worth $30.",
    "Smells AMAZING, however it's too greasy for my oily skin :(",
]})
demo["combined (title = '')"] = demo["raw"].map(lambda t: combine_title_text("", t))
demo["light (LSTM / BERT)"] = demo["combined (title = '')"].map(clean_light)
demo["classical (bag-of-words)"] = demo["combined (title = '')"].map(clean_classical)
print("Auto-title check:", repr(combine_title_text("Five Stars", "Great shampoo")))
display(demo)
print(f"{len(_resources()[0])} stopwords removed; these are deliberately KEPT:\n", sorted(KEEP_WORDS))

### 2.2 Get the data
If the two raw files are already on this computer (in `CFG.local_data_dir`, plain `.jsonl` or gzipped `.jsonl.gz`), they are used directly. Otherwise `hf_hub_download` fetches them from Hugging Face. (The `datasets` loading script for this dataset is broken on recent versions of the `datasets` library, so it is not used.) Downloads are cached, so this is only slow the first time.

In [ ]:
def find_or_download(repo_file: str) -> str:
    name = Path(repo_file).name
    for candidate in (CFG.local_data_dir / name, CFG.local_data_dir / f"{name}.gz"):
        if candidate.exists():
            return str(candidate)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=CFG.hf_repo, filename=repo_file, repo_type="dataset")


t0 = time.time()
review_path = find_or_download(CFG.review_file)
meta_path = find_or_download(CFG.meta_file)
for p in (review_path, meta_path):
    print(f"{Path(p).name:<28} {Path(p).stat().st_size / 1e6:8.1f} MB  ->  {p}")
print(f"Download / cache check took {time.time() - t0:.1f}s")

### 2.3 Load reviews and metadata
Each line of the review file is one JSON object. Only the fields needed are kept: the photo list is reduced to a photo count and the first photo URL (used by the optional image pilot in Section 8d).

In [ ]:
def open_text(path):
    """Open a .jsonl or .jsonl.gz file as text."""
    import gzip
    return gzip.open(path, "rt", encoding="utf-8") if str(path).endswith(".gz") else open(path, "r", encoding="utf-8")


def load_reviews(path) -> pd.DataFrame:
    rows = []
    with open_text(path) as f:
        for line in f:
            r = json.loads(line)
            imgs = r.get("images") or []
            rows.append((r.get("rating"), r.get("title"), r.get("text"), len(imgs),
                         (imgs[0].get("large_image_url") or imgs[0].get("medium_image_url")) if imgs else None,
                         r.get("asin"), r.get("parent_asin"), r.get("user_id"), r.get("timestamp"),
                         r.get("helpful_vote"), r.get("verified_purchase")))
    return pd.DataFrame(rows, columns=["rating", "title", "text", "n_images", "first_image_url", "asin",
                                       "parent_asin", "user_id", "timestamp", "helpful_vote", "verified_purchase"])


def parse_price(p):
    """Metadata price is a float, a string such as '$12.99' or '12.99 - 15.99', or missing."""
    if p is None:
        return np.nan
    if isinstance(p, (int, float)):
        return float(p)
    m = re.search(r"\d+(?:\.\d+)?", str(p).replace(",", ""))
    return float(m.group()) if m else np.nan


def load_meta(path) -> pd.DataFrame:
    rows = []
    with open_text(path) as f:
        for line in f:
            m = json.loads(line)
            rows.append((m.get("parent_asin"), m.get("title"), m.get("main_category"), m.get("store"),
                         parse_price(m.get("price")), m.get("average_rating"), m.get("rating_number")))
    return pd.DataFrame(rows, columns=["parent_asin", "product_title", "main_category", "store",
                                       "price", "average_rating", "rating_number"])


t0 = time.time()
raw = load_reviews(review_path)
meta = load_meta(meta_path)
print(f"Loaded {len(raw):,} reviews and {len(meta):,} product records in {time.time() - t0:.1f}s")
print(f"Memory: reviews {raw.memory_usage(deep=True).sum() / 1e9:.2f} GB | metadata {meta.memory_usage(deep=True).sum() / 1e9:.2f} GB")
display(raw.head(3))
display(meta.head(3))

### 2.4 Clean the text and create labels
Each step is logged with the number of rows before and after.

In [ ]:
log = []   # cleaning log: one row per step


def log_step(step: str, df: pd.DataFrame, note: str = "") -> None:
    before = log[-1]["rows_after"] if log else len(df)
    log.append({"step": step, "rows_before": before, "rows_after": len(df), "removed": before - len(df), "note": note})


df = raw.copy()
log_step("0. Raw file", df, "as downloaded")

# 1. Valid star rating only (1 to 5)
df = df[df["rating"].isin([1, 2, 3, 4, 5])].copy()
df["rating"] = df["rating"].astype(int)
log_step("1. Valid rating", df, "rating must be 1 to 5")

# 2. Strip HTML from title and body, drop auto-generated titles, combine "Title. Body"
n_auto = df["title"].fillna("").map(lambda t: bool(AUTO_TITLE.match(t))).sum()
df["review"] = [combine_title_text(ti, te) for ti, te in zip(df["title"], df["text"])]
log_step("2. Strip HTML + combine title/body", df, f"{n_auto:,} auto-generated titles such as 'Five Stars' removed (label leak)")

# 3. Drop empty reviews (nothing left after cleaning, or fewer than 2 letters)
df = df[df["review"].str.count(r"[A-Za-z]") >= 2].copy()
log_step("3. Drop empty reviews", df, "no usable text after cleaning")

# 4. Drop duplicate records: same reviewer, same product, same text (double submissions / scraping repeats)
df = df.drop_duplicates(subset=["user_id", "parent_asin", "review"]).copy()
log_step("4. Drop duplicate reviews", df, "same user + product + text")

# 5. Labels from stars
LABELS = ["negative", "neutral", "positive"]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
df["label"] = pd.cut(df["rating"], bins=[0, 2, 3, 5], labels=LABELS).astype(str)
df["y"] = df["label"].map(LABEL2ID).astype(int)
log_step("5. Create labels", df, "1-2 = negative, 3 = neutral, 4-5 = positive")

df = df.reset_index(drop=True)
df["review_id"] = np.arange(len(df))     # stable ID used by every split and prediction file
cleaning_log = pd.DataFrame(log)
display(cleaning_log)

TODO-INTERP: cleaning log

**Before and after examples.** The table below shows raw title and body next to the combined, cleaned review for a few reviews that contained HTML or an auto-generated title.

In [ ]:
has_html = df["text"].fillna("").str.contains(r"<br|&amp;|&#\d+;|<[a-z]+>", regex=True)
has_auto = df["title"].fillna("").map(lambda t: bool(AUTO_TITLE.match(t)))
examples = pd.concat([df[has_html].sample(3, random_state=CFG.seed), df[has_auto].sample(2, random_state=CFG.seed)])
with pd.option_context("display.max_colwidth", 160):
    display(examples[["rating", "title", "text", "review"]].rename(columns={"title": "raw title", "text": "raw body", "review": "cleaned review"}))
print(f"Reviews whose raw body contained HTML tags or entities: {has_html.sum():,} ({has_html.mean():.1%})")

### 2.5 Feature engineering
These features are used in the EDA and in the business insights (they are not fed to the text classifiers, which only see text).

| Feature | Why it may matter |
|---|---|
| `char_count`, `word_count`, `sentence_count` | Unhappy customers often explain more |
| `exclaim_count`, `upper_ratio` | Emotion and shouting |
| `has_image`, `n_images` | Photos are evidence: of a great result or of a broken product |
| `verified_purchase` | Verified buyers may be more critical (or more trustworthy) |
| `helpful_vote` | How useful other shoppers found the review |
| `review_date`, `year` | Sentiment trends over time |
| `price`, `price_band`, `store` | From product metadata, joined on `parent_asin` |

In [ ]:
t0 = time.time()
feats = text_features(df["review"])
df = pd.concat([df, feats], axis=1)
df["has_image"] = df["n_images"] > 0
df["verified_purchase"] = df["verified_purchase"].fillna(False).astype(bool)
df["helpful_vote"] = df["helpful_vote"].fillna(0).astype(int)
df["review_date"] = pd.to_datetime(df["timestamp"], unit="ms")
df["year"] = df["review_date"].dt.year

# Join product metadata on parent_asin (asin is a single variant, e.g. one shade; parent_asin is the product)
meta_dedup = meta.drop_duplicates("parent_asin")
df = df.merge(meta_dedup, on="parent_asin", how="left", validate="many_to_one")
PRICE_BINS = [0, 10, 20, 40, np.inf]
PRICE_LABELS = ["< $10", "$10-20", "$20-40", "$40+"]
df["price_band"] = pd.cut(df["price"], bins=PRICE_BINS, labels=PRICE_LABELS, right=False).astype(str).replace("nan", "Unknown")
print(f"Features built in {time.time() - t0:.1f}s")
print(f"Metadata match rate: {df['product_title'].notna().mean():.1%} of reviews | price known for {df['price'].notna().mean():.1%}")
display(df[["review", "label", "char_count", "word_count", "sentence_count", "has_image", "n_images",
            "verified_purchase", "helpful_vote", "review_date", "price", "price_band", "store"]].head())
display(df[["char_count", "word_count", "sentence_count", "n_images", "helpful_vote", "price"]].describe().T.round(2))

TODO-INTERP: features

### 2.6 Two text versions
* **`text_light`** for BiLSTM and DistilBERT: these models read words in order and learn from punctuation, so only URLs and repeated characters are cleaned.
* **`text_classical`** for the bag-of-words models (M1 to M6): lower-cased, contractions expanded, lemmatised (*leaking* becomes *leak*), stopwords removed but negation, contrast and intensity words kept.

The classical version runs in parallel on all CPU cores. The cleaned corpus is cached to `outputs/data/`, so re-runs are fast.

In [ ]:
corpus_cache = CFG.dirs["data"] / "reviews_clean.parquet"
t0 = time.time()
cached = pd.read_parquet(corpus_cache, columns=["review_id", "review", "text_classical"]) if corpus_cache.exists() else None
if cached is not None and len(cached) == len(df) and cached["review"].equals(df["review"]):
    df["text_classical"] = cached["text_classical"].values
    print("Loaded cached classical text.")
else:
    df["text_classical"] = clean_classical_many(df["review"].tolist(), n_jobs=CFG.n_jobs)
df["text_light"] = df["review"].map(clean_light)
print(f"Text versions ready in {time.time() - t0:.1f}s")

# Reviews that are pure stopwords become empty in the classical version; they keep a placeholder token.
n_empty_cls = (df["text_classical"].str.len() == 0).sum()
df.loc[df["text_classical"].str.len() == 0, "text_classical"] = "emptyreview"
print(f"{n_empty_cls:,} reviews ({n_empty_cls / len(df):.2%}) had only stopwords -> placeholder token 'emptyreview'")

keep_cols = ["review_id", "rating", "label", "y", "review", "text_light", "text_classical",
             "char_count", "word_count", "sentence_count", "exclaim_count", "upper_ratio", "has_image", "n_images",
             "first_image_url", "verified_purchase", "helpful_vote", "review_date", "year", "asin", "parent_asin",
             "user_id", "product_title", "store", "price", "price_band", "average_rating", "rating_number", "main_category"]
df = df[keep_cols]
df.to_parquet(corpus_cache, index=False)
del raw; gc.collect()

with pd.option_context("display.max_colwidth", 150):
    display(df.sample(4, random_state=1)[["label", "review", "text_light", "text_classical"]])
print(f"Final cleaned corpus: {len(df):,} reviews x {df.shape[1]} columns (saved to {corpus_cache.name})")

TODO-INTERP: text versions

<a id="s3"></a>
## 3. Exploratory data analysis (full natural distribution)

**Why:** before modelling, we need to know how imbalanced the classes are, whether review length, photos, verified status or helpful votes differ by sentiment, and which words separate the classes. All charts in this section use the **full cleaned corpus**, not the balanced modelling sample, so they describe real customer behaviour.

Colours are used consistently across the notebook: **red = negative, grey = neutral, blue = positive**.

In [ ]:
SENT_COLORS = {"negative": "#e34948", "neutral": "#8f8d86", "positive": "#2a78d6"}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"
plt.rcParams.update({"axes.edgecolor": GRID, "grid.color": GRID, "text.color": INK, "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2, "lines.linewidth": 2, "legend.frameon": False})


def class_share(frame: pd.DataFrame, by: str) -> pd.DataFrame:
    """Row-normalised sentiment shares (%) for each value of `by`, plus the group size."""
    out = pd.crosstab(frame[by], frame["label"], normalize="index")[LABELS] * 100
    out["n_reviews"] = frame[by].value_counts()
    return out.round(1)


def stacked_share_bar(ax, shares: pd.DataFrame, title: str) -> None:
    """100% stacked horizontal bars of sentiment share, with a 2px white gap between segments."""
    left = np.zeros(len(shares))
    for lab in LABELS:
        vals = shares[lab].values
        ax.barh(shares.index.astype(str), vals, left=left, color=SENT_COLORS[lab], label=lab, edgecolor="white", linewidth=2, height=0.6)
        for i, (l, v) in enumerate(zip(left, vals)):
            if v >= 6:
                ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=9, fontweight="bold")
        left += vals
    ax.set_xlim(0, 100); ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(title); ax.grid(axis="y", visible=False)

### 3.1 Class imbalance

In [ ]:
star_counts = df["rating"].value_counts().sort_index()
label_counts = df["label"].value_counts()[LABELS]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
star_col = [SENT_COLORS["negative"]] * 2 + [SENT_COLORS["neutral"]] + [SENT_COLORS["positive"]] * 2
axes[0].bar(star_counts.index.astype(str), star_counts.values, color=star_col, width=0.6)
axes[0].set(title="Reviews by star rating", xlabel="Stars", ylabel="Reviews")
axes[1].bar(LABELS, label_counts.values, color=[SENT_COLORS[l] for l in LABELS], width=0.6)
axes[1].set(title="Reviews by sentiment label", ylabel="Reviews")
for ax, s in [(axes[0], star_counts), (axes[1], label_counts)]:
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v / 1000:.0f}K"))
    ax.grid(axis="x", visible=False)
    for i, v in enumerate(s.values):
        ax.text(i, v, f"{v / s.sum():.1%}", ha="center", va="bottom", fontsize=9, color=INK2)
plt.tight_layout(); savefig(fig, "eda_class_balance"); plt.show()

imbalance = pd.DataFrame({"reviews": label_counts, "share_%": (label_counts / label_counts.sum() * 100).round(1)})
display(imbalance)
print(f"Positive:neutral ratio = {label_counts['positive'] / label_counts['neutral']:.1f} : 1")

TODO-INTERP: class imbalance

### 3.2 Review length vs sentiment
Three views of the same question: box plots (medians labelled), overlaid histograms, and cumulative distributions (CDF). The CDF is the clearest: a curve further to the right means longer reviews.

In [ ]:
len_stats = df.groupby("label")[["word_count", "char_count", "sentence_count"]].agg(["median", "mean"]).loc[LABELS].round(1)
display(len_stats)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
clip = 300
data = [df.loc[df["label"] == l, "word_count"].clip(upper=clip) for l in LABELS]
bp = axes[0].boxplot(data, tick_labels=LABELS, showfliers=False, patch_artist=True, widths=0.5,
                     medianprops={"color": INK, "linewidth": 2})
for patch, l in zip(bp["boxes"], LABELS):
    patch.set_facecolor(SENT_COLORS[l]); patch.set_alpha(0.75)
for i, l in enumerate(LABELS, start=1):
    med = df.loc[df["label"] == l, "word_count"].median()
    axes[0].text(i + 0.28, med, f"median {med:.0f}", va="center", fontsize=9, color=INK)
axes[0].set(title="Words per review (outliers hidden)", ylabel="Words")

bins = np.arange(0, clip + 10, 10)
for l in LABELS:
    axes[1].hist(df.loc[df["label"] == l, "word_count"].clip(upper=clip), bins=bins, density=True,
                 histtype="step", linewidth=2, color=SENT_COLORS[l], label=l)
axes[1].set(title=f"Distribution of words per review (capped at {clip})", xlabel="Words", ylabel="Density")
axes[1].legend()

for l in LABELS:
    x = np.sort(df.loc[df["label"] == l, "word_count"].values)
    axes[2].plot(x, np.arange(1, len(x) + 1) / len(x), color=SENT_COLORS[l], label=l)
axes[2].set(xscale="log", title="Cumulative share of reviews by length", xlabel="Words (log scale)", ylabel="Share of reviews ≤ x")
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1.0)); axes[2].legend()
for q in (0.5,):
    axes[2].axhline(q, color=INK2, lw=0.8, ls="--")
plt.tight_layout(); savefig(fig, "eda_length"); plt.show()

short = (df["word_count"] <= 10).groupby(df["label"]).mean().loc[LABELS] * 100
print("Share of reviews with 10 words or fewer: " + ", ".join(f"{l} {v:.1f}%" for l, v in short.items()))

TODO-INTERP: length

### 3.3 Sentiment over time
Yearly review volume and the share of each sentiment. Years with fewer than 1,000 reviews are left out because their shares are noisy.

In [ ]:
yearly = class_share(df, "year")
yearly = yearly[yearly["n_reviews"] >= 1000]
fig, axes = plt.subplots(1, 2, figsize=(15, 4.3), gridspec_kw={"width_ratios": [1, 1.3]})
axes[0].bar(yearly.index, yearly["n_reviews"], color="#2a78d6", width=0.7)
axes[0].set(title="Reviews per year", ylabel="Reviews"); axes[0].grid(axis="x", visible=False)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v / 1000:.0f}K"))
for l in LABELS:
    axes[1].plot(yearly.index, yearly[l], marker="o", markersize=5, color=SENT_COLORS[l], label=l)
    axes[1].annotate(f"{yearly[l].iloc[-1]:.0f}%", (yearly.index[-1], yearly[l].iloc[-1]), xytext=(6, 0),
                     textcoords="offset points", va="center", fontsize=9, color=INK)
axes[1].set(title="Sentiment share by year", ylabel="Share of reviews"); axes[1].legend(loc="center left")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout(); savefig(fig, "eda_time"); plt.show()
display(yearly.T)

TODO-INTERP: time

### 3.4 Photo vs no photo, and verified vs unverified

In [ ]:
photo = class_share(df.assign(photo=np.where(df["has_image"], "with photo", "no photo")), "photo")
verified = class_share(df.assign(verified=np.where(df["verified_purchase"], "verified", "unverified")), "verified")
fig, axes = plt.subplots(1, 2, figsize=(15, 2.8))
stacked_share_bar(axes[0], photo[LABELS], "Sentiment share: reviews with vs without photos")
stacked_share_bar(axes[1], verified[LABELS], "Sentiment share: verified vs unverified purchases")
axes[1].legend(loc="upper center", bbox_to_anchor=(-0.1, -0.25), ncol=3)
plt.tight_layout(); savefig(fig, "eda_photo_verified"); plt.show()
display(photo); display(verified)
photo_len = df.groupby("has_image")["word_count"].median()
print(f"Median words: with photo {photo_len[True]:.0f} vs without {photo_len[False]:.0f} | "
      f"{df['has_image'].mean():.1%} of reviews have a photo, {df['verified_purchase'].mean():.1%} are verified")

TODO-INTERP: photo verified

### 3.5 Helpful votes by sentiment

In [ ]:
helpful = df.groupby("label")["helpful_vote"].agg(
    mean_votes="mean", median_votes="median",
    share_with_1plus=lambda s: (s >= 1).mean() * 100, share_with_5plus=lambda s: (s >= 5).mean() * 100).loc[LABELS].round(2)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].bar(LABELS, helpful["mean_votes"], color=[SENT_COLORS[l] for l in LABELS], width=0.55)
axes[0].set(title="Average helpful votes per review", ylabel="Votes")
axes[1].bar(LABELS, helpful["share_with_1plus"], color=[SENT_COLORS[l] for l in LABELS], width=0.55)
axes[1].set(title="Share of reviews with at least 1 helpful vote", ylabel="Share of reviews")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
for ax, col, fmt in [(axes[0], "mean_votes", "{:.2f}"), (axes[1], "share_with_1plus", "{:.1f}%")]:
    ax.grid(axis="x", visible=False)
    for i, v in enumerate(helpful[col]):
        ax.text(i, v, fmt.format(v), ha="center", va="bottom", fontsize=9, color=INK)
plt.tight_layout(); savefig(fig, "eda_helpful"); plt.show()
display(helpful)

TODO-INTERP: helpful

### 3.6 Words and phrases that distinguish each class
Raw word counts are dominated by common words ("great", "hair"). Instead, each word or two-word phrase is scored with the **weighted log-odds ratio with an informative Dirichlet prior** (Monroe, Colaresi and Quinn, 2008). It compares how often a phrase appears in one class versus the other two, and shrinks rare phrases towards zero so they cannot top the list by chance. A z-score above about 2 means the difference is unlikely to be noise. It runs on a balanced random sample (equal reviews per class) of the classical text.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer


def log_odds_dirichlet(counts: np.ndarray, y: np.ndarray, vocab: np.ndarray, top: int = 15) -> dict:
    """Monroe et al. (2008) weighted log-odds with an informative Dirichlet prior, one class vs the rest."""
    prior = np.asarray(counts.sum(axis=0)).ravel() + 0.01   # the whole corpus acts as the prior
    a0 = prior.sum()
    out = {}
    for k, lab in enumerate(LABELS):
        yi = np.asarray(counts[y == k].sum(axis=0)).ravel()
        yj = np.asarray(counts[y != k].sum(axis=0)).ravel()
        ni, nj = yi.sum(), yj.sum()
        delta = (np.log((yi + prior) / (ni + a0 - yi - prior)) - np.log((yj + prior) / (nj + a0 - yj - prior)))
        var = 1 / (yi + prior) + 1 / (yj + prior)
        z = delta / np.sqrt(var)
        idx = np.argsort(-z)[:top]
        out[lab] = pd.DataFrame({"phrase": vocab[idx], "z": z[idx].round(1), "count_in_class": yi[idx].astype(int)})
    return out


eda_sample = df.groupby("label", group_keys=False).sample(n=min(40_000, label_counts.min()), random_state=CFG.seed)
cv_eda = CountVectorizer(ngram_range=(1, 2), min_df=20, token_pattern=r"\b\w+\b")
X_eda = cv_eda.fit_transform(eda_sample["text_classical"])
top_phrases = log_odds_dirichlet(X_eda, eda_sample["y"].values, cv_eda.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(16, 5.2))
for ax, lab in zip(axes, LABELS):
    t = top_phrases[lab].iloc[::-1]
    ax.barh(t["phrase"], t["z"], color=SENT_COLORS[lab], height=0.65)
    ax.set(title=f"Most distinctive of {lab} reviews", xlabel="log-odds z-score"); ax.grid(axis="y", visible=False)
plt.tight_layout(); savefig(fig, "eda_distinctive_phrases"); plt.show()
print(f"Sample: {len(eda_sample):,} reviews, vocabulary {X_eda.shape[1]:,} unigrams + bigrams (min_df=20)")
display(pd.concat({lab: top_phrases[lab].head(10).reset_index(drop=True) for lab in LABELS}, axis=1))

TODO-INTERP: distinctive phrases

**EDA summary: what this means for modelling**

TODO-INTERP: EDA summary

<a id="s4"></a>
## 4. Split design

**The design**
1. **Modelling pool.** Reviews with identical text (after lower-casing) are kept once, so the same text cannot appear in both training and test data. Short reviews such as "Love it!" are repeated thousands of times, and copies across splits would inflate test scores.
2. **Balanced sample.** The same number of reviews is drawn from each class (capped by `CFG.n_per_class` and by the smallest class), then split **70 / 15 / 15** into train / validation / test, stratified by label.
3. **Natural-mix test set.** A separate random sample from the rest of the pool, with the real class mix (mostly positive). No review in it is in the balanced sample.

**Why both test sets?**
* A balanced training set stops the models from learning "always say positive". With roughly 7 positive reviews for every neutral one, that shortcut would score well on accuracy while missing most neutral reviews.
* A balanced test set gives a fair per-class comparison between models.
* The **natural-mix** test set shows what the business would actually see in production: accuracy on real traffic and how many false alarms the minority classes get.

**Rules that keep the evaluation honest** (the example notebook broke several of these):
* Every model uses the **same** splits. They are saved to `outputs/data/` and re-loaded, never re-drawn.
* Hyperparameters, decision thresholds and ensemble weights are chosen on **validation only**.
* Each model is scored on the test sets **once**, after all its choices are fixed.

In [ ]:
split_file = CFG.dirs["data"] / "split_assignment.parquet"
split_meta_file = CFG.dirs["data"] / "split_config.json"
split_cfg = {"seed": CFG.seed, "n_per_class": CFG.n_per_class, "split_fracs": list(CFG.split_fracs),
             "n_natural_test": CFG.n_natural_test, "n_reviews_clean": len(df)}

# 1. Modelling pool: one row per distinct (lower-cased, whitespace-normalised) review text
text_key = df["review"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
pool = df[~text_key.duplicated(keep="first")]
print(f"Modelling pool: {len(pool):,} reviews ({len(df) - len(pool):,} repeated texts removed, e.g. 'Love it!')")
print(pool["label"].value_counts()[LABELS].to_string())

if split_file.exists() and split_meta_file.exists() and load_json(split_meta_file) == split_cfg:
    assign = pd.read_parquet(split_file)
    print("\nLoaded saved split assignment (same config) -> identical splits to earlier runs.")
else:
    from sklearn.model_selection import train_test_split
    # 2. Balanced sample, capped by the smallest class
    n_per_class = min(CFG.n_per_class, pool["label"].value_counts().min())
    balanced = pool.groupby("label", group_keys=False).sample(n=n_per_class, random_state=CFG.seed)
    f_tr, f_va, f_te = CFG.split_fracs
    tr_ids, rest_ids = train_test_split(balanced["review_id"], train_size=f_tr, stratify=balanced["label"], random_state=CFG.seed)
    rest = balanced.set_index("review_id").loc[rest_ids]
    va_ids, te_ids = train_test_split(rest.index, test_size=f_te / (f_va + f_te), stratify=rest["label"], random_state=CFG.seed)
    # 3. Natural-mix test set from the rest of the pool
    remaining = pool[~pool["review_id"].isin(balanced["review_id"])]
    nat_ids = remaining.sample(n=CFG.n_natural_test, random_state=CFG.seed)["review_id"]
    assign = pd.concat([pd.DataFrame({"review_id": ids, "split": s}) for s, ids in
                        [("train", tr_ids), ("val", va_ids), ("test", te_ids), ("natural", nat_ids)]], ignore_index=True)
    assign.to_parquet(split_file, index=False)
    save_json(split_cfg, split_meta_file)
    print(f"\nNew split assignment saved to {split_file.name} ({n_per_class:,} reviews per class in the balanced sample)")

split_of = assign.set_index("review_id")["split"]
data = df.set_index("review_id").loc[assign["review_id"]].reset_index()
data["split"] = data["review_id"].map(split_of)
train_df, val_df, test_df, nat_df = (data[data["split"] == s].reset_index(drop=True) for s in ["train", "val", "test", "natural"])

# Save each split (text + label + features) so any later section, or another tool, can reuse it
for name, part in [("train", train_df), ("val", val_df), ("test", test_df), ("natural", nat_df)]:
    part.to_parquet(CFG.dirs["data"] / f"split_{name}.parquet", index=False)

# Sanity checks: no overlap between splits, by ID or by text
sets = {s: set(p["review_id"]) for s, p in [("train", train_df), ("val", val_df), ("test", test_df), ("natural", nat_df)]}
assert all(sets[a].isdisjoint(sets[b]) for a in sets for b in sets if a < b), "review IDs overlap between splits"
texts = {s: set(p["review"].str.lower()) for s, p in [("train", train_df), ("val", val_df), ("test", test_df), ("natural", nat_df)]}
assert all(texts[a].isdisjoint(texts[b]) for a in texts for b in texts if a < b), "identical text in two splits"

split_table = pd.crosstab(data["split"], data["label"]).loc[["train", "val", "test", "natural"], LABELS]
split_table["total"] = split_table.sum(axis=1)
split_table["neutral_%"] = (split_table["neutral"] / split_table["total"] * 100).round(1)
display(split_table)
print("No review ID or review text appears in more than one split.")

TODO-INTERP: splits

<a id="s5"></a>
## 5. One evaluation function for every model

**Why:** the example notebook copy-pasted the same plotting block for every model, so small differences crept in. Here **one function, `evaluate_model`, is used by every model**, so every model is measured and plotted in exactly the same way. It:

1. prints the classification report (precision, recall, F1 per class);
2. draws a six-panel dashboard: **confusion matrix**, **one-vs-rest ROC curves**, **precision-recall curves**, **confidence histogram** (correct vs wrong predictions), **neutral-class calibration** (does "70% neutral" really mean neutral 70% of the time?) and per-class precision / recall / F1;
3. records accuracy, macro F1, per-class F1, ROC-AUC, log-loss, neutral calibration error, **training time** and **inference time** in a shared results table (`outputs/results/results.csv`);
4. saves the predicted probabilities to `outputs/predictions/`.

In [ ]:
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report, confusion_matrix,
                             f1_score, log_loss, precision_recall_curve, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)

class ResultsBook:
    """Shared results table with one row per (model, split), saved to CSV after every update."""

    def __init__(self, path):
        self.path = Path(path)
        self.df = pd.read_csv(self.path) if self.path.exists() else pd.DataFrame()

    def record(self, row: dict) -> None:
        if len(self.df):
            self.df = self.df[~((self.df["model"] == row["model"]) & (self.df["split"] == row["split"]))]
        self.df = pd.concat([self.df, pd.DataFrame([row])], ignore_index=True)
        self.df.to_csv(self.path, index=False)

    def table(self, split: str, sort_by: str = "macro_f1") -> pd.DataFrame:
        t = self.df[self.df["split"] == split].sort_values(sort_by, ascending=False)
        return t.reset_index(drop=True)


def neutral_calibration(y_true, proba, n_bins: int = 10):
    """Reliability table for P(neutral): predicted probability vs observed neutral rate, plus ECE."""
    p, hit = proba[:, 1], (np.asarray(y_true) == 1).astype(float)
    edges = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, n_bins - 1)
    rows = [(p[idx == b].mean(), hit[idx == b].mean(), (idx == b).sum()) for b in range(n_bins) if (idx == b).any()]
    cal = pd.DataFrame(rows, columns=["mean_predicted", "observed_rate", "n"])
    ece = float((cal["n"] / len(p) * (cal["mean_predicted"] - cal["observed_rate"]).abs()).sum())
    return cal, ece


def _dashboard(name, split, y, proba, pred, metrics, cal):
    fig, axes = plt.subplots(2, 3, figsize=(17, 9.6))
    # (1) confusion matrix: counts and row %
    ax = axes[0, 0]
    cm = confusion_matrix(y, pred, labels=[0, 1, 2])
    cm_pct = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:,}\n({cm_pct[i, j]:.0%})", ha="center", va="center",
                    color="white" if cm_pct[i, j] > 0.55 else INK, fontsize=10)
    ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS,
           xlabel="Predicted", ylabel="Actual", title="Confusion matrix (row %)")
    ax.grid(False)
    # (2) ROC and (3) precision-recall, one-vs-rest
    for k, lab in enumerate(LABELS):
        fpr, tpr, _ = roc_curve(y == k, proba[:, k])
        axes[0, 1].plot(fpr, tpr, color=SENT_COLORS[lab], label=f"{lab} (AUC {metrics[f'auc_{lab}']:.3f})")
        prec, rec, _ = precision_recall_curve(y == k, proba[:, k])
        axes[0, 2].plot(rec, prec, color=SENT_COLORS[lab], label=f"{lab} (AP {metrics[f'ap_{lab}']:.3f})")
    axes[0, 1].plot([0, 1], [0, 1], ls="--", color=INK2, lw=1)
    axes[0, 1].set(title="ROC curves (one-vs-rest)", xlabel="False positive rate", ylabel="True positive rate")
    axes[0, 2].set(title="Precision-recall curves", xlabel="Recall", ylabel="Precision", ylim=(0, 1.02))
    for ax in (axes[0, 1], axes[0, 2]):
        ax.legend(loc="lower left" if ax is axes[0, 2] else "lower right", fontsize=9)
    # (4) confidence histogram
    conf, correct = proba.max(axis=1), pred == y
    bins = np.linspace(1 / 3, 1, 21)
    for mask, lab, col in [(correct, "correct", "#2a78d6"), (~correct, "wrong", "#eb6834")]:
        if mask.any():
            axes[1, 0].hist(conf[mask], bins=bins, alpha=0.75, color=col, label=f"{lab} (n={mask.sum():,}, mean {conf[mask].mean():.2f})")
    axes[1, 0].set(title="Confidence of predictions", xlabel="Highest class probability", ylabel="Reviews")
    axes[1, 0].legend(fontsize=9)
    # (5) neutral calibration
    ax = axes[1, 1]
    ax.plot([0, 1], [0, 1], ls="--", color=INK2, lw=1, label="perfect calibration")
    ax.plot(cal["mean_predicted"], cal["observed_rate"], marker="o", markersize=7, color=SENT_COLORS["neutral"], label="model")
    ax.set(title=f"Neutral-class calibration (ECE {metrics['neutral_ece']:.3f})", xlabel="Predicted P(neutral)",
           ylabel="Observed share neutral", xlim=(0, 1), ylim=(0, 1))
    ax.legend(fontsize=9)
    # (6) per-class precision / recall / F1
    ax = axes[1, 2]
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    x = np.arange(3)
    for off, vals, lab, col in [(-0.26, p, "precision", "#9ec5f4"), (0, r, "recall", "#3987e5"), (0.26, f, "F1", "#184f95")]:
        ax.bar(x + off, vals, width=0.24, color=col, label=lab)
        for xi, v in zip(x + off, vals):
            ax.text(xi, v + 0.01, f"{v:.2f}", ha="center", fontsize=8, color=INK)
    ax.set(xticks=x, xticklabels=LABELS, ylim=(0, 1.1), title="Precision, recall and F1 by class")
    ax.legend(fontsize=9, ncol=3, loc="upper center")
    ax.grid(axis="x", visible=False)
    fig.suptitle(f"{name}  |  {split} set (n={len(y):,})  |  accuracy {metrics['accuracy']:.3f}  |  macro F1 {metrics['macro_f1']:.3f}",
                 fontsize=14, fontweight="bold", y=1.0)
    fig.tight_layout()
    return fig


def _compact(name, split, y, pred, metrics):
    fig, ax = plt.subplots(figsize=(4.6, 3.9))
    cm = confusion_matrix(y, pred, labels=[0, 1, 2])
    cm_pct = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{cm[i, j]:,}\n({cm_pct[i, j]:.0%})", ha="center", va="center",
                    color="white" if cm_pct[i, j] > 0.55 else INK, fontsize=9)
    ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS, xlabel="Predicted", ylabel="Actual")
    ax.set_title(f"{name}: {split} set\nacc {metrics['accuracy']:.3f} | macro F1 {metrics['macro_f1']:.3f}", fontsize=10)
    ax.grid(False)
    fig.tight_layout()
    return fig


def evaluate_model(name, y_true, proba, *, split, book=None, ids=None, family="", train_time=np.nan,
                   infer_time=np.nan, plots="full", fig_dir=None, pred_dir=None, report=True, notes=""):
    """Score one model on one split.

    plots: "full" (six-panel dashboard), "compact" (confusion matrix only) or None.
    Returns a dict of metrics, which is also recorded in `book` (a ResultsBook) if given.
    """
    y = np.asarray(y_true).astype(int)
    proba = np.clip(np.asarray(proba, dtype=float), 1e-7, 1)
    proba = proba / proba.sum(axis=1, keepdims=True)
    pred = proba.argmax(axis=1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1, 2], zero_division=0)
    cal, ece = neutral_calibration(y, proba)
    m = {"model": name, "family": family, "split": split, "n": len(y),
         "accuracy": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
         "weighted_f1": f1_score(y, pred, average="weighted")}
    for k, lab in enumerate(LABELS):
        m[f"f1_{lab}"], m[f"precision_{lab}"], m[f"recall_{lab}"] = f[k], p[k], r[k]
        m[f"auc_{lab}"] = roc_auc_score(y == k, proba[:, k])
        m[f"ap_{lab}"] = average_precision_score(y == k, proba[:, k])
    m["roc_auc_macro"] = float(np.mean([m[f"auc_{l}"] for l in LABELS]))
    m["log_loss"] = log_loss(y, proba, labels=[0, 1, 2])
    m["neutral_ece"] = ece
    m["train_time_s"] = train_time
    m["infer_time_s"] = infer_time
    m["infer_ms_per_review"] = infer_time / len(y) * 1000 if infer_time == infer_time else np.nan
    m["notes"] = notes

    if report:
        print(f"=== {name} | {split} set (n={len(y):,}) ===")
        print(classification_report(y, pred, labels=[0, 1, 2], target_names=LABELS, digits=3))
    print(f"[{split:>7}] {name}: accuracy {m['accuracy']:.3f} | macro F1 {m['macro_f1']:.3f} | "
          f"F1 neg/neu/pos {m['f1_negative']:.3f}/{m['f1_neutral']:.3f}/{m['f1_positive']:.3f} | "
          f"inference {m['infer_ms_per_review']:.3f} ms/review")
    if plots:
        fig = _dashboard(name, split, y, proba, pred, m, cal) if plots == "full" else _compact(name, split, y, pred, m)
        if fig_dir is not None:
            safe = "".join(c if c.isalnum() else "_" for c in name)
            fig.savefig(Path(fig_dir) / f"eval_{safe}_{split}.png", bbox_inches="tight")
        plt.show()
    if pred_dir is not None and ids is not None:
        safe = "".join(c if c.isalnum() else "_" for c in name)
        pd.DataFrame({"review_id": ids, "y_true": y, "y_pred": pred, **{f"p_{l}": proba[:, k] for k, l in enumerate(LABELS)}}
                     ).to_parquet(Path(pred_dir) / f"{safe}__{split}.parquet", index=False)
    if book is not None:
        book.record(m)
    return m


def plot_top_features(feature_names, weights, title, top: int = 12, ax_row=None):
    """Bar charts of the features that push most strongly towards each class (one panel per class).

    weights: array (3, n_features); larger = stronger evidence for that class.
    """
    names = np.asarray(feature_names)
    fig, axes = plt.subplots(1, 3, figsize=(16, 0.32 * top + 1.6))
    for k, (ax, lab) in enumerate(zip(axes, LABELS)):
        w = weights[k] - np.delete(weights, k, axis=0).mean(axis=0)   # evidence for k relative to the other two
        idx = np.argsort(-w)[:top][::-1]
        ax.barh(names[idx], w[idx], color=SENT_COLORS[lab], height=0.65)
        ax.set_title(f"Pushes towards {lab}", fontsize=11)
        ax.grid(axis="y", visible=False)
        ax.tick_params(axis="y", labelsize=9)
    fig.suptitle(title, fontweight="bold")
    fig.tight_layout()
    return fig

In [ ]:
BOOK = ResultsBook(CFG.dirs["results"] / "results.csv")
SPLITS = {"val": val_df, "test": test_df, "natural": nat_df}


def report(name: str, predict_proba, columns, *, family: str, train_time: float, notes: str = "",
           test_plots: str = "full", natural_plots=None) -> dict:
    """Predict on validation, balanced test and natural test; time inference; evaluate and record each.

    predict_proba: function taking a DataFrame/Series of inputs and returning an (n, 3) probability array.
    Validation results are recorded silently (used later for model choice and ensemble weights).
    """
    out = {}
    for split, frame in SPLITS.items():
        X = frame[columns]
        t0 = time.perf_counter()
        proba = predict_proba(X)
        t_inf = time.perf_counter() - t0
        out[split] = proba
        evaluate_model(name, frame["y"].values, proba, split=split, book=BOOK, ids=frame["review_id"].values,
                          family=family, train_time=train_time, infer_time=t_inf, notes=notes,
                          plots={"val": None, "test": test_plots, "natural": natural_plots}[split],
                          report=(split == "test"), fig_dir=CFG.dirs["figures"], pred_dir=CFG.dirs["predictions"])
    return out


# Smoke test of the evaluation function on random probabilities (not a real model) to show it works before any model runs.
_rng = np.random.default_rng(0)
_y = _rng.integers(0, 3, 600)
_p = _rng.dirichlet([1, 1, 1], 600) * 0.5 + np.eye(3)[_y] * 0.5
_ = evaluate_model("Smoke test (random)", _y, _p, split="demo", plots=None, report=False)
print("Evaluation function ready. Results are recorded to", BOOK.path)

<a id="s6"></a>
## 6. Models

Eleven models, from a no-training word list (M0) to a fine-tuned transformer (M9) and an ensemble (M10). Each one has a short "what it is and why I'm trying it" note before it and an interpretation after it.

**Shared tools for the classical models**
* `tune_on_validation`: scikit-learn's `RandomizedSearchCV` with a **`PredefinedSplit`**, so the only "fold" is *fit on train, score macro F1 on validation*. The winning settings are then re-fitted on the training split only, which keeps the validation predictions honest for choosing ensemble weights later.
* `fit_or_load`: saves each fitted model to `outputs/models/` with its training time and settings. If `CFG.force_retrain_classical` is False, the saved model is loaded instead of retrained.

In [ ]:
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier

y_train, y_val = train_df["y"].values, val_df["y"].values


def tune_on_validation(pipe, param_dist, columns, n_iter, search_jobs=CFG.n_jobs):
    """Random search scored by macro F1 on the validation split, then refit the winner on train only."""
    X_all = pd.concat([train_df[columns], val_df[columns]], ignore_index=True)
    y_all = np.concatenate([y_train, y_val])
    fold = PredefinedSplit(np.r_[np.full(len(train_df), -1), np.zeros(len(val_df))])
    search = RandomizedSearchCV(pipe, param_dist, n_iter=n_iter, scoring="f1_macro", cv=fold, refit=False,
                                random_state=CFG.seed, n_jobs=search_jobs, error_score="raise")
    t0 = time.time(); search.fit(X_all, y_all); search_time = time.time() - t0
    best = clone(pipe).set_params(**search.best_params_)
    t0 = time.time(); best.fit(train_df[columns], y_train); fit_time = time.time() - t0
    results = (pd.DataFrame(search.cv_results_)
               .filter(regex=r"^param_|mean_test_score|mean_fit_time")
               .rename(columns=lambda c: c.replace("param_", "").replace("mean_test_score", "val_macro_f1"))
               .sort_values("val_macro_f1", ascending=False).reset_index(drop=True))
    print(f"Searched {n_iter} settings in {search_time:.0f}s; best validation macro F1 = {search.best_score_:.4f}; "
          f"final fit on train took {fit_time:.1f}s")
    return best, search.best_params_, results, search_time, fit_time


def fit_or_load(name, build_fn, force=CFG.force_retrain_classical):
    """Train via build_fn() -> (model, info dict) or load a saved copy. Returns (model, info)."""
    path, info_path = CFG.dirs["models"] / f"{name}.joblib", CFG.dirs["models"] / f"{name}.json"
    if path.exists() and info_path.exists() and not force:
        print(f"Loaded saved {name} from {path.name}")
        return joblib.load(path), load_json(info_path)
    model, info = build_fn()
    joblib.dump(model, path, compress=3)
    save_json(info, info_path)
    print(f"Saved {name} -> {path.name} ({path.stat().st_size / 1e6:.1f} MB)")
    return model, info


def show_search(results, top=5):
    with pd.option_context("display.float_format", "{:.4f}".format):
        display(results.head(top))

### M0. VADER lexicon baseline (no training)
**What it is:** VADER is a hand-built dictionary of about 7,500 words and emoticons, each with a human-rated sentiment score, plus rules for negation ("not good"), intensifiers ("very"), capital letters and exclamation marks. It returns a *compound* score from −1 (very negative) to +1 (very positive).

**Why try it:** it needs no training data, so it is the cheapest thing the business could deploy tomorrow. Every other model has to beat it to justify its cost. The usual cut-offs (±0.05) are designed for social media, so the two cut-offs (negative below *a*, positive above *b*) are **tuned on the validation set**. That is the only fitting step. Probabilities for the ROC and calibration plots are a smooth function of the distance to the cut-offs, so they are *scores*, not true probabilities.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

_vader = SentimentIntensityAnalyzer()


def vader_compound(texts) -> np.ndarray:
    return np.array([_vader.polarity_scores(t)["compound"] for t in texts])


def vader_proba(compound, t_neg, t_pos, temp=0.1) -> np.ndarray:
    """Turn the compound score into 3 class scores whose arg-max equals the threshold rule."""
    s = np.c_[t_neg - compound, np.minimum(compound - t_neg, t_pos - compound), compound - t_pos] / temp
    e = np.exp(s - s.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


from sklearn.metrics import f1_score
t0 = time.time()
c_val = vader_compound(val_df["review"])
grid = [(a, b) for a in np.arange(-0.9, 0.85, 0.05) for b in np.arange(-0.85, 0.96, 0.05) if b > a]
scores = [(a, b, f1_score(y_val, vader_proba(c_val, a, b).argmax(1), average="macro")) for a, b in grid]
vader_grid = pd.DataFrame(scores, columns=["t_neg", "t_pos", "val_macro_f1"]).sort_values("val_macro_f1", ascending=False)
T_NEG, T_POS = vader_grid.iloc[0][["t_neg", "t_pos"]].round(2)
m0_time = time.time() - t0
default_f1 = f1_score(y_val, vader_proba(c_val, -0.05, 0.05).argmax(1), average="macro")
print(f"Default cut-offs (-0.05 / +0.05): validation macro F1 = {default_f1:.3f}")
print(f"Tuned cut-offs ({T_NEG:+.2f} / {T_POS:+.2f}): validation macro F1 = {vader_grid.iloc[0]['val_macro_f1']:.3f}")
save_json({"t_neg": T_NEG, "t_pos": T_POS, "train_time_s": m0_time}, CFG.dirs["models"] / "M0_VADER.json")

fig, ax = plt.subplots(figsize=(9, 3.6))
for k, lab in enumerate(LABELS):
    ax.hist(c_val[y_val == k], bins=40, histtype="step", linewidth=2, color=SENT_COLORS[lab], label=lab, density=True)
for t in (T_NEG, T_POS):
    ax.axvline(t, color=INK, ls="--", lw=1)
ax.set(title="VADER compound score by true class (validation) with tuned cut-offs", xlabel="compound score", ylabel="density")
ax.legend(); plt.show()

m0 = report("M0 VADER", lambda s: vader_proba(vader_compound(s), T_NEG, T_POS), "review",
            family="Lexicon", train_time=m0_time, notes=f"cut-offs {T_NEG:+.2f}/{T_POS:+.2f} tuned on validation")

TODO-INTERP: M0

### M1. CountVectorizer + chi-square + Logistic Regression
**What it is:** the review becomes a vector of word and phrase counts (bag of words). A **chi-square test** keeps the *k* phrases most associated with the labels, and **logistic regression** learns one weight per phrase per class.

**Why try it:** it is the classic, fully transparent text classifier: each weight can be read directly ("*refund* pushes towards negative"). Random search tunes the n-gram range, minimum document frequency, binary vs raw counts, the number of features kept and the regularisation strength *C*.

In [ ]:
def build_m1():
    pipe = Pipeline([
        ("vec", CountVectorizer(token_pattern=r"\b\w+\b")),
        ("sel", SelectKBest(chi2)),
        ("clf", LogisticRegression(max_iter=3000)),
    ])
    params = {"vec__ngram_range": [(1, 1), (1, 2), (1, 3)], "vec__min_df": [2, 3, 5], "vec__binary": [True, False],
              "sel__k": [5_000, 10_000, 20_000, 40_000], "clf__C": loguniform(0.02, 5)}
    model, best, res, t_search, t_fit = tune_on_validation(pipe, params, "text_classical", CFG.n_search_iter)
    show_search(res)
    return model, {"best_params": best, "train_time_s": t_fit, "search_time_s": t_search}


m1_model, m1_info = fit_or_load("M1_Count_chi2_LR", build_m1)
m1 = report("M1 Count+chi2+LR", m1_model.predict_proba, "text_classical", family="Classical",
            train_time=m1_info["train_time_s"], notes=str(m1_info["best_params"]))

TODO-INTERP: M1

**What drives M1's decisions?** The weights of the selected phrases, shown relative to the other two classes.

In [ ]:
m1_names = m1_model.named_steps["vec"].get_feature_names_out()[m1_model.named_steps["sel"].get_support()]
fig = plot_top_features(m1_names, m1_model.named_steps["clf"].coef_, "M1: phrases with the largest logistic-regression weights")
savefig(fig, "features_M1"); plt.show()

TODO-INTERP: M1 features

### M2. TF-IDF on words *and* characters + Logistic Regression
**What it is:** two views of the same review, joined side by side:
* **word TF-IDF** on the classical text (1 to 3-word phrases). TF-IDF down-weights words that appear in almost every review;
* **character n-gram TF-IDF** (2 to 5 characters) on the *light* text, which still has punctuation and unlemmatised words. Character pieces catch misspellings ("terible"), word stems ("irritat-") and emphasis ("!!!") that the word view loses.

**Why try it:** combining a "meaning" view and a "spelling" view of the text usually beats either alone, and it stays a linear, explainable model.

In [ ]:
def build_m2():
    features = ColumnTransformer([
        ("word", TfidfVectorizer(token_pattern=r"\b\w+\b", sublinear_tf=True), "text_classical"),
        ("char", TfidfVectorizer(analyzer="char_wb", sublinear_tf=True, min_df=3, dtype=np.float32), "text_light"),
    ])
    pipe = Pipeline([("features", features), ("clf", LogisticRegression(max_iter=3000))])
    params = {"features__word__ngram_range": [(1, 2), (1, 3)], "features__word__min_df": [2, 3],
              "features__char__ngram_range": [(2, 5), (3, 5), (2, 4)], "features__char__max_features": [100_000, 200_000],
              "clf__C": loguniform(0.5, 20)}
    model, best, res, t_search, t_fit = tune_on_validation(pipe, params, ["text_classical", "text_light"],
                                                           max(4, CFG.n_search_iter // 2))
    show_search(res)
    return model, {"best_params": best, "train_time_s": t_fit, "search_time_s": t_search}


m2_model, m2_info = fit_or_load("M2_TFIDF_word_char_LR", build_m2)
m2 = report("M2 TF-IDF word+char LR", m2_model.predict_proba, ["text_classical", "text_light"], family="Classical",
            train_time=m2_info["train_time_s"], notes=str(m2_info["best_params"]))

In [ ]:
m2_names = np.array([n.replace("word__", "").replace("char__", "[c]") for n in m2_model.named_steps["features"].get_feature_names_out()])
fig = plot_top_features(m2_names, m2_model.named_steps["clf"].coef_,
                           "M2: strongest word and [c]haracter n-gram features (character n-grams marked [c], spaces shown as ' ')", top=15)
savefig(fig, "features_M2"); plt.show()

TODO-INTERP: M2

### M3. TF-IDF + Complement Naive Bayes
**What it is:** Naive Bayes (named in the brief) treats every word as independent evidence and multiplies the evidence together. The **Complement** variant estimates each class's word weights from all the *other* classes, which is more stable for text and usually beats standard multinomial Naive Bayes.

**Why try it:** it trains in about a second, needs very little memory, and is a strong baseline for short texts.

In [ ]:
def build_m3():
    pipe = Pipeline([("vec", TfidfVectorizer(token_pattern=r"\b\w+\b", sublinear_tf=True)), ("clf", ComplementNB())])
    params = {"vec__ngram_range": [(1, 1), (1, 2), (1, 3)], "vec__min_df": [1, 2, 3, 5],
              "clf__alpha": loguniform(0.01, 3), "clf__norm": [False, True]}
    model, best, res, t_search, t_fit = tune_on_validation(pipe, params, "text_classical", CFG.n_search_iter)
    show_search(res)
    return model, {"best_params": best, "train_time_s": t_fit, "search_time_s": t_search}


m3_model, m3_info = fit_or_load("M3_TFIDF_ComplementNB", build_m3)
m3 = report("M3 TF-IDF+ComplementNB", m3_model.predict_proba, "text_classical", family="Classical",
            train_time=m3_info["train_time_s"], notes=str(m3_info["best_params"]))

In [ ]:
fig = plot_top_features(m3_model.named_steps["vec"].get_feature_names_out(), m3_model.named_steps["clf"].feature_log_prob_,
                           "M3: words with the highest Complement-NB weight for each class")
savefig(fig, "features_M3"); plt.show()

TODO-INTERP: M3

### M4. TF-IDF + Linear SVM (calibrated)
**What it is:** a linear Support Vector Machine finds the boundary that separates the classes with the widest margin. SVMs output scores, not probabilities, so the SVM is wrapped in `CalibratedClassifierCV` (3-fold, sigmoid), which learns to turn scores into probabilities on held-out folds of the *training* data.

**Why try it:** linear SVMs are often the single strongest classical model for text, and calibrated probabilities let it join the same ROC, calibration and ensemble analysis as everything else.

In [ ]:
def build_m4():
    pipe = Pipeline([("vec", TfidfVectorizer(token_pattern=r"\b\w+\b", sublinear_tf=True)),
                     ("clf", CalibratedClassifierCV(LinearSVC(random_state=CFG.seed), method="sigmoid", cv=3))])
    params = {"vec__ngram_range": [(1, 2), (1, 3)], "vec__min_df": [1, 2, 3],
              "clf__estimator__C": loguniform(0.02, 2)}
    model, best, res, t_search, t_fit = tune_on_validation(pipe, params, "text_classical", CFG.n_search_iter)
    show_search(res)
    return model, {"best_params": best, "train_time_s": t_fit, "search_time_s": t_search}


m4_model, m4_info = fit_or_load("M4_TFIDF_LinearSVC", build_m4)
m4 = report("M4 TF-IDF+LinearSVC", m4_model.predict_proba, "text_classical", family="Classical",
            train_time=m4_info["train_time_s"], notes=str(m4_info["best_params"]))

In [ ]:
svm_coef = np.mean([c.estimator.coef_ for c in m4_model.named_steps["clf"].calibrated_classifiers_], axis=0)
fig = plot_top_features(m4_model.named_steps["vec"].get_feature_names_out(), svm_coef,
                           "M4: largest linear-SVM weights (averaged over the 3 calibration folds)")
savefig(fig, "features_M4"); plt.show()

TODO-INTERP: M4

### M5. TF-IDF + chi-square + Random Forest
**What it is:** hundreds of decision trees, each trained on a random subset of reviews and features, voting together. Chi-square first keeps the most label-related phrases, because trees struggle with tens of thousands of sparse columns.

**Why try it:** trees can capture interactions ("*smell*" is bad only together with "*chemical*") that linear models miss. It also tests whether the business needs anything beyond linear models. Only a few settings are searched because each fit takes minutes on this machine.

In [ ]:
def build_m5():
    pipe = Pipeline([("vec", TfidfVectorizer(token_pattern=r"\b\w+\b", sublinear_tf=True, ngram_range=(1, 2), min_df=3)),
                     ("sel", SelectKBest(chi2)),
                     ("clf", RandomForestClassifier(n_jobs=CFG.n_jobs, random_state=CFG.seed))])
    params = {"sel__k": [3_000, 6_000, 10_000], "clf__n_estimators": [300, 500],
              "clf__min_samples_leaf": [1, 2, 3], "clf__max_features": ["sqrt", 0.02]}
    model, best, res, t_search, t_fit = tune_on_validation(pipe, params, "text_classical", CFG.n_search_iter_trees, search_jobs=1)
    show_search(res)
    return model, {"best_params": best, "train_time_s": t_fit, "search_time_s": t_search}


if CFG.run_tree_models:
    m5_model, m5_info = fit_or_load("M5_TFIDF_chi2_RandomForest", build_m5)
    m5 = report("M5 TF-IDF+chi2+RandomForest", m5_model.predict_proba, "text_classical", family="Classical",
                train_time=m5_info["train_time_s"], notes=str(m5_info["best_params"]))

TODO-INTERP: M5

### M6. TF-IDF + chi-square + XGBoost
**What it is:** gradient-boosted trees: each new tree corrects the mistakes of the trees before it. XGBoost handles sparse inputs efficiently and adds regularisation.

**Why try it:** boosting is usually the strongest tree method on tabular data. If it cannot beat logistic regression on text, that tells us the problem is mostly linear in the words.

In [ ]:
from xgboost import XGBClassifier


def build_m6():
    pipe = Pipeline([("vec", TfidfVectorizer(token_pattern=r"\b\w+\b", sublinear_tf=True, ngram_range=(1, 2), min_df=3)),
                     ("sel", SelectKBest(chi2)),
                     ("clf", XGBClassifier(objective="multi:softprob", tree_method="hist", n_jobs=CFG.n_jobs,
                                           random_state=CFG.seed, eval_metric="mlogloss"))])
    params = {"sel__k": [5_000, 10_000, 20_000], "clf__n_estimators": [300, 600], "clf__max_depth": [4, 6, 8],
              "clf__learning_rate": [0.05, 0.1, 0.2], "clf__subsample": [0.8, 1.0], "clf__colsample_bytree": [0.3, 0.6]}
    model, best, res, t_search, t_fit = tune_on_validation(pipe, params, "text_classical", CFG.n_search_iter_trees, search_jobs=1)
    show_search(res)
    return model, {"best_params": best, "train_time_s": t_fit, "search_time_s": t_search}


if CFG.run_tree_models:
    m6_model, m6_info = fit_or_load("M6_TFIDF_chi2_XGBoost", build_m6)
    m6 = report("M6 TF-IDF+chi2+XGBoost", m6_model.predict_proba, "text_classical", family="Classical",
                train_time=m6_info["train_time_s"], notes=str(m6_info["best_params"]))

In [ ]:
if CFG.run_tree_models:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    for ax, (mdl, title) in zip(axes, [(m5_model, "M5 Random Forest: impurity importance"), (m6_model, "M6 XGBoost: gain importance")]):
        names = mdl.named_steps["vec"].get_feature_names_out()[mdl.named_steps["sel"].get_support()]
        imp = mdl.named_steps["clf"].feature_importances_
        idx = np.argsort(-imp)[:20][::-1]
        ax.barh(names[idx], imp[idx], color="#2a78d6", height=0.65)
        ax.set(title=title, xlabel="importance (share of total)"); ax.grid(axis="y", visible=False)
    plt.tight_layout(); savefig(fig, "features_trees"); plt.show()

TODO-INTERP: M5 M6 features

#### Classical models so far (validation set)
The validation scores decide which linear model joins the ensemble in M10. The test sets are not used for that choice.

In [ ]:
val_board = BOOK.table("val")[["model", "accuracy", "macro_f1", "f1_negative", "f1_neutral", "f1_positive", "train_time_s"]]
display(val_board.style.format(precision=3).background_gradient(subset=["macro_f1"], cmap="Blues"))
LINEAR_CANDIDATES = {"M1 Count+chi2+LR": m1_model, "M2 TF-IDF word+char LR": m2_model,
                     "M3 TF-IDF+ComplementNB": m3_model, "M4 TF-IDF+LinearSVC": m4_model}
BEST_LINEAR = val_board[val_board["model"].isin(LINEAR_CANDIDATES)].iloc[0]["model"]
print(f"Best linear model on VALIDATION macro F1: {BEST_LINEAR}")

### Deep learning set-up (shared by M7, M8 and M9)
All neural models use **PyTorch** only, so the same code runs on an NVIDIA GPU (CUDA), an Apple Silicon Mac (MPS) or a CPU. The shared code in the next cell provides:
* a simple word tokeniser and vocabulary for the LSTMs;
* a **length-bucketed batch sampler**: reviews of similar length are batched together, so far less padding is processed (a large speed-up on CPU);
* the two LSTM architectures;
* one training loop with **progress printing, early stopping on validation macro F1, and a checkpoint of the best epoch** to `outputs/models/`.

In [ ]:
# PyTorch models and training utilities shared by the BiLSTM, attention-BiLSTM and DistilBERT models.
import math
from collections import Counter

import torch.nn as nn
from sklearn.metrics import f1_score
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

_LSTM_TOKEN = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[!?]")


def word_tokenize(text: str) -> list:
    """Lower-case word tokens; '!' and '?' kept as tokens because they carry emotion."""
    return _LSTM_TOKEN.findall(text.lower())


class Vocab:
    PAD, UNK = 0, 1

    def __init__(self, itos):
        self.itos = list(itos)
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    @classmethod
    def build(cls, token_lists, max_size=30_000, min_freq=2):
        counts = Counter(t for toks in token_lists for t in toks)
        words = [w for w, c in counts.most_common(max_size - 2) if c >= min_freq]
        return cls(["<pad>", "<unk>"] + words)

    def encode(self, tokens, max_len):
        ids = [self.stoi.get(t, self.UNK) for t in tokens[:max_len]]
        return ids or [self.UNK]

    def save(self, path):
        with open(path, "w") as f:
            json.dump(self.itos, f)

    @classmethod
    def load(cls, path):
        with open(path) as f:
            return cls(json.load(f))

    def __len__(self):
        return len(self.itos)


class BucketBatches:
    """Yields lists of indices; each batch holds reviews of similar length (less padding = faster)."""

    def __init__(self, lengths, batch_size, shuffle, seed=0):
        self.lengths, self.bs, self.shuffle, self.seed, self.epoch = np.asarray(lengths), batch_size, shuffle, seed, 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        idx = rng.permutation(len(self.lengths)) if self.shuffle else np.arange(len(self.lengths))
        chunk = self.bs * 50
        batches = []
        for s in range(0, len(idx), chunk):
            part = idx[s:s + chunk]
            part = part[np.argsort(self.lengths[part], kind="stable")]
            batches += [part[i:i + self.bs] for i in range(0, len(part), self.bs)]
        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return math.ceil(len(self.lengths) / self.bs)


def make_batches(seqs, labels, batch_size, shuffle, seed=0):
    """Generator of padded batches (dicts of tensors) from lists of token-id sequences."""
    sampler = BucketBatches([len(s) for s in seqs], batch_size, shuffle, seed)

    class _Loader:
        def __len__(self):
            return len(sampler)

        def __iter__(self):
            for b in sampler:
                lens = [len(seqs[i]) for i in b]
                ids = torch.zeros(len(b), max(lens), dtype=torch.long)
                for r, i in enumerate(b):
                    ids[r, :lens[r]] = torch.tensor(seqs[i])
                batch = {"input_ids": ids, "attention_mask": (ids != 0).long(), "index": torch.tensor(b)}
                if labels is not None:
                    batch["labels"] = torch.tensor(np.asarray(labels)[b], dtype=torch.long)
                yield batch

    return _Loader()


class BiLSTMClassifier(nn.Module):
    """Embedding -> bidirectional LSTM -> [mean-pool ; max-pool] -> dropout -> linear."""

    def __init__(self, vocab_size, emb_dim=128, hidden=128, dropout=0.4, n_classes=3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(4 * hidden, n_classes)

    def encode(self, input_ids, attention_mask):
        lengths = attention_mask.sum(1).clamp(min=1)
        x = self.drop(self.emb(input_ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=input_ids.size(1))
        return out, lengths

    def forward(self, input_ids, attention_mask, **_):
        out, lengths = self.encode(input_ids, attention_mask)
        mask = attention_mask.unsqueeze(-1).bool()
        mean = (out * mask).sum(1) / lengths.unsqueeze(1)
        mx = out.masked_fill(~mask, -1e4).max(1).values
        return self.fc(self.drop(torch.cat([mean, mx], dim=1)))


class AttentionBiLSTMClassifier(BiLSTMClassifier):
    """BiLSTM whose word outputs are combined by learned attention weights (one weight per word)."""

    def __init__(self, vocab_size, emb_dim=128, hidden=128, dropout=0.4, n_classes=3, attn_dim=128):
        super().__init__(vocab_size, emb_dim, hidden, dropout, n_classes)
        self.attn = nn.Sequential(nn.Linear(2 * hidden, attn_dim), nn.Tanh(), nn.Linear(attn_dim, 1, bias=False))
        self.fc = nn.Linear(2 * hidden, n_classes)

    def forward(self, input_ids, attention_mask, return_attention=False, **_):
        out, _ = self.encode(input_ids, attention_mask)
        scores = self.attn(out).squeeze(-1).masked_fill(attention_mask == 0, -1e4)
        alpha = torch.softmax(scores, dim=1)                       # (batch, words), sums to 1 per review
        context = (alpha.unsqueeze(-1) * out).sum(1)
        logits = self.fc(self.drop(context))
        return (logits, alpha) if return_attention else logits


class HFClassifier(nn.Module):
    """Wrap a Hugging Face sequence-classification model so it returns plain logits."""

    def __init__(self, hf_model):
        super().__init__()
        self.hf = hf_model

    def forward(self, input_ids, attention_mask, **_):
        return self.hf(input_ids=input_ids, attention_mask=attention_mask).logits


@torch.no_grad()
def torch_predict_proba(model, loader, device):
    """Class probabilities in the ORIGINAL order of the inputs (batches are length-sorted internally)."""
    model.eval()
    out, order = [], []
    for batch in loader:
        logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
        out.append(torch.softmax(logits.float(), -1).cpu().numpy())
        order.append(batch["index"].numpy())
    probs, order = np.concatenate(out), np.concatenate(order)
    res = np.zeros_like(probs)
    res[order] = probs
    return res


def train_classifier(model, train_loader, val_loader, y_val, *, epochs, lr, ckpt_path, device,
                     weight_decay=0.01, warmup_frac=0.0, patience=2, grad_clip=1.0, evals_per_epoch=1, log=print):
    """AdamW + linear warm-up/decay, early stopping on validation macro F1, best weights saved to ckpt_path."""
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total = epochs * len(train_loader)
    warm = int(warmup_frac * total)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / max(1, warm) if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    loss_fn = nn.CrossEntropyLoss()
    eval_at = {int(len(train_loader) * (k + 1) / evals_per_epoch) for k in range(evals_per_epoch)}
    best_f1, bad, step, history, t0 = -1.0, 0, 0, [], time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, n_seen = 0.0, 0
        for i, batch in enumerate(train_loader, start=1):
            logits = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            loss = loss_fn(logits, batch["labels"].to(device))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            sched.step()
            step += 1
            run_loss += loss.item() * len(batch["labels"])
            n_seen += len(batch["labels"])
            if i % max(1, len(train_loader) // 5) == 0 and i not in eval_at:
                log(f"  epoch {epoch} | batch {i}/{len(train_loader)} | train loss {run_loss / n_seen:.4f} | {time.time() - t0:.0f}s")
            if i in eval_at:
                proba = torch_predict_proba(model, val_loader, device)
                f1 = f1_score(y_val, proba.argmax(1), average="macro")
                vloss = float(-np.log(np.clip(proba[np.arange(len(y_val)), y_val], 1e-9, 1)).mean())
                improved = f1 > best_f1
                history.append({"epoch": round(epoch - 1 + i / len(train_loader), 2), "train_loss": run_loss / n_seen,
                                "val_loss": vloss, "val_macro_f1": f1, "elapsed_s": time.time() - t0})
                log(f"  epoch {history[-1]['epoch']:.2f} | train loss {run_loss / n_seen:.4f} | val loss {vloss:.4f} | "
                    f"val macro F1 {f1:.4f}{'  <- best, saved' if improved else ''} | {time.time() - t0:.0f}s")
                if improved:
                    best_f1, bad = f1, 0
                    torch.save(model.state_dict(), ckpt_path)
                else:
                    bad += 1
                model.train()
                if bad >= patience:
                    log(f"  early stopping: no improvement in {patience} evaluations")
                    return history, time.time() - t0
    return history, time.time() - t0

In [ ]:
DEVICE = torch.device(CFG.device)


def plot_history(hist: pd.DataFrame, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
    axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", color="#2a78d6", label="train loss")
    axes[0].plot(hist["epoch"], hist["val_loss"], marker="o", color="#eb6834", label="validation loss")
    axes[0].set(title=f"{title}: loss", xlabel="epoch"); axes[0].legend()
    axes[1].plot(hist["epoch"], hist["val_macro_f1"], marker="o", color="#1baf7a")
    best = hist["val_macro_f1"].idxmax()
    axes[1].annotate(f"best {hist.loc[best, 'val_macro_f1']:.3f}", (hist.loc[best, "epoch"], hist.loc[best, "val_macro_f1"]),
                     xytext=(0, -16), textcoords="offset points", ha="center", fontsize=9)
    axes[1].set(title=f"{title}: validation macro F1", xlabel="epoch")
    plt.tight_layout(); plt.show()


# ---- LSTM inputs: vocabulary built on the TRAINING split only -------------------------------
vocab_path = CFG.dirs["models"] / "lstm_vocab.json"
train_tokens = [word_tokenize(t) for t in train_df["text_light"]]
if vocab_path.exists() and not CFG.force_retrain_deep:
    vocab = Vocab.load(vocab_path)
else:
    vocab = Vocab.build(train_tokens, max_size=CFG.lstm_vocab_size, min_freq=2)
    vocab.save(vocab_path)


def lstm_encode(texts):
    return [vocab.encode(word_tokenize(t), CFG.lstm_max_len) for t in texts]


seq_train = [vocab.encode(t, CFG.lstm_max_len) for t in train_tokens]
seq_val = lstm_encode(val_df["text_light"])
lens = np.array([len(t) for t in train_tokens])
print(f"Vocabulary: {len(vocab):,} words (train split, min frequency 2) | "
      f"review length in tokens: median {np.median(lens):.0f}, 95th percentile {np.percentile(lens, 95):.0f}; "
      f"{(lens > CFG.lstm_max_len).mean():.1%} truncated at {CFG.lstm_max_len}")


def lstm_predictor(model):
    def _predict(texts):
        return torch_predict_proba(model, make_batches(lstm_encode(texts), None, 256, shuffle=False), DEVICE)
    return _predict


def fit_or_load_torch(name, model, train_loader, val_loader, train_kwargs):
    """Train with checkpointing, or load the saved best checkpoint if it exists (and retraining is off)."""
    ckpt, info_path = CFG.dirs["models"] / f"{name}.pt", CFG.dirs["models"] / f"{name}.json"
    if ckpt.exists() and info_path.exists() and not CFG.force_retrain_deep:
        print(f"Loading saved checkpoint {ckpt.name} (training skipped; set CFG.force_retrain_deep=True to retrain)")
        info = load_json(info_path)
    else:
        seed_everything()
        hist, t_train = train_classifier(model, train_loader, val_loader, y_val, ckpt_path=ckpt, device=DEVICE, **train_kwargs)
        info = {"train_time_s": t_train, "history": hist, "params": {k: str(v) for k, v in train_kwargs.items()}}
        save_json(info, info_path)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.to(DEVICE).eval()
    return model, info

### M7. Bidirectional LSTM
**What it is:** an LSTM reads the review one word at a time and keeps a running "memory", so word order matters: "not bad at all" is read in sequence instead of as a bag of words. *Bidirectional* means a second LSTM reads the review backwards, so every word sees context from both sides. The word outputs are summarised by average- and max-pooling, then classified. Word embeddings are learned from scratch on the training split.

**Why try it:** it is the first model here that understands word order. If it beats M2, order matters; if not, bag-of-words already captures most of the signal.

In [ ]:
if CFG.run_lstm:
    seed_everything()
    m7_net = BiLSTMClassifier(len(vocab))
    print(f"M7 parameters: {sum(p.numel() for p in m7_net.parameters()):,}")
    m7_net, m7_info = fit_or_load_torch(
        "M7_BiLSTM", m7_net,
        make_batches(seq_train, y_train, CFG.lstm_batch, shuffle=True, seed=CFG.seed),
        make_batches(seq_val, None, 256, shuffle=False),
        dict(epochs=CFG.lstm_epochs, lr=2e-3, weight_decay=1e-4, patience=2))
    plot_history(pd.DataFrame(m7_info["history"]), "M7 BiLSTM")
    m7 = report("M7 BiLSTM", lstm_predictor(m7_net), "text_light", family="Recurrent NN",
                train_time=m7_info["train_time_s"], notes="embeddings learned from scratch")

TODO-INTERP: M7

### M8. Bidirectional LSTM with attention
**What it is:** the same BiLSTM, but instead of averaging all word outputs it learns an **attention weight** for each word and takes a weighted average. Words that matter for the decision (e.g. "returned", "rash") should get high weight.

**Why try it:** attention gives a built-in explanation, a heat map of which words the model looked at, which the plain BiLSTM cannot provide. Accuracy is not expected to jump: the point is explainability.

In [ ]:
if CFG.run_lstm:
    seed_everything()
    m8_net = AttentionBiLSTMClassifier(len(vocab))
    print(f"M8 parameters: {sum(p.numel() for p in m8_net.parameters()):,}")
    m8_net, m8_info = fit_or_load_torch(
        "M8_BiLSTM_Attention", m8_net,
        make_batches(seq_train, y_train, CFG.lstm_batch, shuffle=True, seed=CFG.seed),
        make_batches(seq_val, None, 256, shuffle=False),
        dict(epochs=CFG.lstm_epochs, lr=2e-3, weight_decay=1e-4, patience=2))
    plot_history(pd.DataFrame(m8_info["history"]), "M8 BiLSTM + attention")
    m8 = report("M8 BiLSTM+Attention", lstm_predictor(m8_net), "text_light", family="Recurrent NN",
                train_time=m8_info["train_time_s"], notes="attention pooling")

TODO-INTERP: M8

**Attention heat maps.** Each row is one test review. Darker cells are the words the model weighted most when deciding. Reviews are chosen to show one correct prediction per class plus mistakes.

In [ ]:
@torch.no_grad()
def attention_for(text):
    tokens = word_tokenize(text)[:CFG.lstm_max_len]
    ids = torch.tensor([vocab.encode(tokens, CFG.lstm_max_len)], device=DEVICE)
    logits, alpha = m8_net(ids, (ids != 0).long(), return_attention=True)
    return tokens, alpha[0].cpu().numpy(), torch.softmax(logits, -1)[0].cpu().numpy()


def plot_attention(examples, max_words=28):
    fig, axes = plt.subplots(len(examples), 1, figsize=(16, 1.05 * len(examples)))
    for ax, (text, true) in zip(np.atleast_1d(axes), examples):
        toks, a, p = attention_for(text)
        toks, a = toks[:max_words], a[:max_words]
        ax.imshow(a[None, :], cmap="Oranges", aspect="auto", vmin=0, vmax=max(a.max(), 1e-6))
        for j, t in enumerate(toks):
            ax.text(j, 0, t, ha="center", va="center", fontsize=8.5, rotation=0,
                    color="white" if a[j] > 0.6 * a.max() else INK)
        ax.set_yticks([]); ax.set_xticks([]); ax.grid(False)
        ax.set_title(f"true: {true} | predicted: {LABELS[p.argmax()]} ({p.max():.0%})" + ("" if len(word_tokenize(text)) <= max_words else "  [first words only]"),
                     fontsize=9, loc="left")
    plt.tight_layout(); savefig(fig, "attention_heatmaps"); plt.show()


if CFG.run_lstm:
    m8_test_pred = pd.read_parquet(CFG.dirs["predictions"] / "M8_BiLSTM_Attention__test.parquet").merge(
        test_df[["review_id", "text_light", "word_count"]], on="review_id")
    short = m8_test_pred[m8_test_pred["word_count"].between(8, 26)]
    picks = []
    for k in range(3):   # one confident, correct example per class
        cand = short[(short.y_true == k) & (short.y_pred == k)].sort_values(f"p_{LABELS[k]}", ascending=False)
        if len(cand):
            picks.append(cand.iloc[min(3, len(cand) - 1)])
    for wrong in [(short.y_true == 2) & (short.y_pred == 0), (short.y_true == 1) & (short.y_pred != 1)]:   # two mistakes
        if wrong.any():
            picks.append(short[wrong].iloc[0])
    plot_attention([(r.text_light, LABELS[r.y_true]) for r in picks])

TODO-INTERP: attention

### M9. DistilBERT, fine-tuned
**What it is:** DistilBERT is a smaller, 40% faster version of Google's BERT transformer (66 million parameters, 6 layers). It was pre-trained on English Wikipedia and books to understand language in context. Every word attends to every other word at once, so "not" is linked directly to the word it negates however far apart they are. We add a 3-class output layer and **fine-tune** the whole network on our reviews.

**Why try it:** pre-training gives the model knowledge of English that the other models must learn from our 25,000 training reviews alone. Transformers are the current standard for text classification, and the brief names BERT.

**Settings and the CPU budget.** DistilBERT is by far the slowest model on this machine (no GPU). To keep the notebook within about three hours, it fine-tunes on a **stratified subset of the same training split** (`CFG.bert_train_n` reviews), with reviews cut at `CFG.bert_max_len` word-pieces. It is evaluated on the **same** validation and test sets as every other model. Training uses a linear learning-rate warm-up and decay, checks validation macro F1 twice per epoch, keeps the best checkpoint and stops early if validation stops improving. The checkpoint is saved in half precision, split into files under 50 MB, so later runs (and Section 9) load it instead of retraining.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

BERT_DIR = CFG.dirs["models"] / "M9_DistilBERT"


def bert_predictor(model, tokenizer, batch_size=64):
    def _predict(texts):
        enc = tokenizer(list(texts), truncation=True, max_length=CFG.bert_max_len)["input_ids"]
        return torch_predict_proba(model, make_batches(enc, None, batch_size, shuffle=False), DEVICE)
    return _predict


if CFG.run_bert:
    if (BERT_DIR / "config.json").exists() and not CFG.force_retrain_deep:
        print(f"Loading fine-tuned DistilBERT from {BERT_DIR} (training skipped)")
        bert_tok = AutoTokenizer.from_pretrained(BERT_DIR)
        m9_net = HFClassifier(AutoModelForSequenceClassification.from_pretrained(BERT_DIR, dtype=torch.float32))
        m9_info = load_json(BERT_DIR / "training_info.json")
    else:
        seed_everything()
        bert_tok = AutoTokenizer.from_pretrained(CFG.bert_name)
        assert bert_tok.pad_token_id == 0, "the batching code pads with id 0"
        hf = AutoModelForSequenceClassification.from_pretrained(
            CFG.bert_name, num_labels=3, id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)
        m9_net = HFClassifier(hf)
        bert_train = (train_df.groupby("label", group_keys=False)
                      .sample(n=min(CFG.bert_train_n, len(train_df)) // 3, random_state=CFG.seed))
        enc_train = bert_tok(bert_train["text_light"].tolist(), truncation=True, max_length=CFG.bert_max_len)["input_ids"]
        enc_val = bert_tok(val_df["text_light"].tolist(), truncation=True, max_length=CFG.bert_max_len)["input_ids"]
        tl = np.array([len(e) for e in enc_train])
        print(f"Fine-tuning on {len(bert_train):,} training reviews ({bert_train['label'].value_counts().to_dict()}) | "
              f"word-pieces: median {np.median(tl):.0f}, {np.mean(tl == CFG.bert_max_len):.1%} truncated at {CFG.bert_max_len}")
        ckpt = CFG.dirs["models"] / "M9_DistilBERT_best.pt"
        hist, t_train = train_classifier(
            m9_net, make_batches(enc_train, bert_train["y"].values, CFG.bert_batch, shuffle=True, seed=CFG.seed),
            make_batches(enc_val, None, 64, shuffle=False), y_val,
            epochs=CFG.bert_epochs, lr=CFG.bert_lr, weight_decay=0.01, warmup_frac=0.1, patience=2,
            evals_per_epoch=2, ckpt_path=ckpt, device=DEVICE)
        m9_net.load_state_dict(torch.load(ckpt, map_location="cpu"))
        # Save the best epoch as a standard Hugging Face folder: fp16, sharded into files < 50 MB.
        m9_net.hf.half().save_pretrained(BERT_DIR, max_shard_size="45MB")
        bert_tok.save_pretrained(BERT_DIR)
        m9_info = {"train_time_s": t_train, "history": hist, "train_n": len(bert_train), "max_len": CFG.bert_max_len,
                   "epochs": CFG.bert_epochs, "lr": CFG.bert_lr, "batch": CFG.bert_batch}
        save_json(m9_info, BERT_DIR / "training_info.json")
        ckpt.unlink()
        # Reload from the saved folder so every reported number comes from the saved model.
        m9_net = HFClassifier(AutoModelForSequenceClassification.from_pretrained(BERT_DIR, dtype=torch.float32))
    m9_net.to(DEVICE).eval()
    print(f"DistilBERT parameters: {sum(p.numel() for p in m9_net.parameters()):,} | training time {m9_info['train_time_s'] / 60:.1f} min")
    plot_history(pd.DataFrame(m9_info["history"]), "M9 DistilBERT")
    m9 = report("M9 DistilBERT", bert_predictor(m9_net, bert_tok), "text_light", family="Transformer",
                train_time=m9_info["train_time_s"], notes=f"fine-tuned on {m9_info['train_n']:,} reviews, max_len {m9_info['max_len']}")

TODO-INTERP: M9

**Why did DistilBERT decide that? Word-occlusion importance.** Transformers have no simple weights to read. Instead, each word is removed in turn and the review is re-scored: if the predicted-class probability drops a lot, that word mattered. Red words support the prediction; blue words argue against it. The reviews are the same ones used for the M8 attention heat maps, so the two explanations can be compared.

In [ ]:
@torch.no_grad()
def occlusion_importance(text, predict):
    words = text.split()
    variants = [text] + [" ".join(words[:i] + words[i + 1:]) for i in range(len(words))]
    p = predict(variants)
    k = p[0].argmax()
    return words, p[0][k] - p[1:, k], k, p[0]


def plot_occlusion(examples, predict, max_words=30):
    fig, axes = plt.subplots(len(examples), 1, figsize=(16, 1.05 * len(examples)))
    for ax, (text, true) in zip(np.atleast_1d(axes), examples):
        words, imp, k, p0 = occlusion_importance(" ".join(text.split()[:max_words]), predict)
        lim = max(np.abs(imp).max(), 0.02)
        ax.imshow(imp[None, :], cmap="RdBu_r", aspect="auto", vmin=-lim, vmax=lim)
        for j, w in enumerate(words):
            ax.text(j, 0, w, ha="center", va="center", fontsize=8.5, color="white" if abs(imp[j]) > 0.6 * lim else INK)
        ax.set_yticks([]); ax.set_xticks([]); ax.grid(False)
        ax.set_title(f"true: {true} | predicted: {LABELS[k]} ({p0[k]:.0%}) | most important word: '{words[int(np.argmax(imp))]}' "
                     f"(removing it drops P({LABELS[k]}) by {imp.max():.2f})", fontsize=9, loc="left")
    plt.tight_layout(); savefig(fig, "bert_occlusion"); plt.show()


if CFG.run_bert:
    if CFG.run_lstm:
        occl_examples = [(r.text_light, LABELS[r.y_true]) for r in picks]
    else:
        occl_examples = [(t, l) for t, l in test_df.sample(5, random_state=CFG.seed)[["text_light", "label"]].values]
    plot_occlusion(occl_examples, bert_predictor(m9_net, bert_tok, batch_size=32))

TODO-INTERP: occlusion

### M10. Weighted soft-vote ensemble: DistilBERT + best linear model
**What it is:** the final probability is a weighted average, *w* × P(DistilBERT) + (1 − *w*) × P(best linear model). The linear model is whichever of M1 to M4 scored best on **validation** (chosen in the classical-model summary above).

**Why try it:** the two models make different kinds of mistakes. DistilBERT reads context; the linear model reads specific words and spellings and was trained on all 25,200 training reviews, not a subset. Averaging their probabilities often fixes cases where one is unsure. The weight *w* is chosen **on the validation set only**. (The example notebook tuned its ensemble weights on the test set, which makes its ensemble score optimistic.)

The ensemble re-uses the probabilities both models already produced, so its inference time is the **sum** of the two models' times.

In [ ]:
if CFG.run_bert:
    safe = lambda n: "".join(c if c.isalnum() else "_" for c in n)
    load_p = lambda name, split: pd.read_parquet(CFG.dirs["predictions"] / f"{safe(name)}__{split}.parquet")
    P = {split: {"bert": load_p("M9 DistilBERT", split), "lin": load_p(BEST_LINEAR, split)} for split in SPLITS}
    for split in SPLITS:   # both files must list the same reviews in the same order
        assert (P[split]["bert"]["review_id"].values == P[split]["lin"]["review_id"].values).all()
    pcols = [f"p_{l}" for l in LABELS]

    weights = np.round(np.arange(0, 1.0001, 0.05), 2)
    val_curve = pd.DataFrame({"w_bert": weights, "val_macro_f1": [
        f1_score(y_val, (w * P["val"]["bert"][pcols].values + (1 - w) * P["val"]["lin"][pcols].values).argmax(1), average="macro")
        for w in weights]})
    W_BERT = float(val_curve.loc[val_curve["val_macro_f1"].idxmax(), "w_bert"])
    save_json({"w_bert": W_BERT, "linear_model": BEST_LINEAR}, CFG.dirs["models"] / "M10_ensemble.json")

    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.plot(val_curve["w_bert"], val_curve["val_macro_f1"], marker="o", markersize=5, color="#2a78d6")
    ax.axvline(W_BERT, color=INK2, ls="--", lw=1)
    ax.annotate(f"chosen w = {W_BERT:.2f}\nval macro F1 {val_curve['val_macro_f1'].max():.3f}", (W_BERT, val_curve["val_macro_f1"].max()),
                xytext=(8, -28), textcoords="offset points", fontsize=9)
    ax.set(title=f"Choosing the ensemble weight on VALIDATION (DistilBERT + {BEST_LINEAR})",
           xlabel="weight on DistilBERT (0 = linear only, 1 = DistilBERT only)", ylabel="validation macro F1")
    plt.tight_layout(); savefig(fig, "ensemble_weight"); plt.show()

    lin_rows = BOOK.df[BOOK.df["model"] == BEST_LINEAR].set_index("split")
    bert_rows = BOOK.df[BOOK.df["model"] == "M9 DistilBERT"].set_index("split")
    m10 = {}
    for split, frame in SPLITS.items():
        proba = W_BERT * P[split]["bert"][pcols].values + (1 - W_BERT) * P[split]["lin"][pcols].values
        m10[split] = proba
        evaluate_model("M10 Ensemble", frame["y"].values, proba, split=split, book=BOOK, ids=frame["review_id"].values,
                          family="Transformer", train_time=bert_rows.loc["test", "train_time_s"] + lin_rows.loc["test", "train_time_s"],
                          infer_time=bert_rows.loc[split, "infer_time_s"] + lin_rows.loc[split, "infer_time_s"],
                          plots={"val": None, "test": "full", "natural": None}[split], report=(split == "test"),
                          notes=f"{W_BERT:.2f} x DistilBERT + {1 - W_BERT:.2f} x {BEST_LINEAR}",
                          fig_dir=CFG.dirs["figures"], pred_dir=CFG.dirs["predictions"])

TODO-INTERP: M10

<a id="s7"></a>
## 7. Leaderboard and deployment recommendation

All numbers below come from the shared results table that `evaluate_model` filled in. Nothing is typed in by hand.

In [ ]:
LB_COLS = {"model": "model", "family": "family", "accuracy": "accuracy", "macro_f1": "macro F1", "f1_negative": "F1 neg",
           "f1_neutral": "F1 neu", "f1_positive": "F1 pos", "roc_auc_macro": "ROC-AUC", "neutral_ece": "neutral ECE",
           "train_time_s": "train s", "infer_ms_per_review": "infer ms/review"}


def leaderboard(split):
    t = BOOK.table(split)[list(LB_COLS)].rename(columns=LB_COLS)
    t.insert(0, "rank", np.arange(1, len(t) + 1))
    t["min to score 1M reviews"] = t["infer ms/review"] * 1e6 / 1000 / 60
    return t


def show_board(t, title):
    display(Markdown(f"**{title}**"))
    display(t.style.format({c: "{:.3f}" for c in ["accuracy", "macro F1", "F1 neg", "F1 neu", "F1 pos", "ROC-AUC", "neutral ECE"]}
                            | {"train s": "{:,.0f}", "infer ms/review": "{:.3f}", "min to score 1M reviews": "{:,.1f}"})
            .background_gradient(subset=["macro F1", "F1 neu"], cmap="Blues").hide(axis="index"))


lb_test, lb_nat = leaderboard("test"), leaderboard("natural")
show_board(lb_test, f"Balanced test set (n = {len(test_df):,}, one-third per class)")
show_board(lb_nat, f"Natural-mix test set (n = {len(nat_df):,}, {nat_df['label'].value_counts(normalize=True)['positive']:.0%} positive)")
lb_test.to_csv(CFG.dirs["results"] / "leaderboard_balanced_test.csv", index=False)
lb_nat.to_csv(CFG.dirs["results"] / "leaderboard_natural_test.csv", index=False)

TODO-INTERP: leaderboard tables

In [ ]:
FAMILY_COLORS = {"Lexicon": "#8f8d86", "Classical": "#2a78d6", "Recurrent NN": "#eb6834", "Transformer": "#1baf7a"}

fig, axes = plt.subplots(1, 2, figsize=(17, 6.2), gridspec_kw={"width_ratios": [1.15, 1]})
# (a) per-class F1 on the balanced test set
order = lb_test.sort_values("macro F1")
y = np.arange(len(order))
for off, lab in zip([-0.27, 0, 0.27], LABELS):
    axes[0].barh(y + off, order[f"F1 {lab[:3]}"], height=0.26, color=SENT_COLORS[lab], label=lab)
axes[0].set(yticks=y, yticklabels=order["model"], xlim=(0.3, 1), xlabel="F1 score (balanced test set)",
            title="Per-class F1 by model (sorted by macro F1)")
axes[0].legend(loc="lower right"); axes[0].grid(axis="y", visible=False)
# (b) accuracy vs speed
for fam, grp in lb_test.groupby("family"):
    axes[1].scatter(grp["infer ms/review"], grp["macro F1"], s=90, color=FAMILY_COLORS[fam], label=fam,
                    edgecolor="white", linewidth=1.5, zorder=3)
for _, r in lb_test.iterrows():
    axes[1].annotate(r["model"].split(" ")[0], (r["infer ms/review"], r["macro F1"]), xytext=(6, 4),
                     textcoords="offset points", fontsize=9, color=INK)
axes[1].set(xscale="log", xlabel="Inference time per review, ms (log scale, this machine)", ylabel="Macro F1 (balanced test)",
            title="Accuracy vs speed: up and to the left is better")
axes[1].legend(title="Model family", loc="lower right")
plt.tight_layout(); savefig(fig, "leaderboard_charts"); plt.show()

TODO-INTERP: leaderboard charts

**How the leaders behave on real traffic.** Confusion matrices on the natural-mix test set for the best classical model, DistilBERT and the ensemble (read from the saved prediction files).

In [ ]:
def plot_cm_grid(models, split):
    from sklearn.metrics import confusion_matrix
    fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 4.3))
    for ax, name in zip(np.atleast_1d(axes), models):
        pr = pd.read_parquet(CFG.dirs["predictions"] / f"{''.join(c if c.isalnum() else '_' for c in name)}__{split}.parquet")
        cm = confusion_matrix(pr["y_true"], pr["y_pred"], labels=[0, 1, 2])
        pct = cm / cm.sum(axis=1, keepdims=True)
        ax.imshow(pct, cmap="Blues", vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{cm[i, j]:,}\n({pct[i, j]:.0%})", ha="center", va="center", fontsize=9,
                        color="white" if pct[i, j] > 0.55 else INK)
        row = BOOK.df[(BOOK.df["model"] == name) & (BOOK.df["split"] == split)].iloc[0]
        ax.set(xticks=range(3), yticks=range(3), xticklabels=LABELS, yticklabels=LABELS, xlabel="Predicted", ylabel="Actual",
               title=f"{name}\nacc {row['accuracy']:.3f} | macro F1 {row['macro_f1']:.3f}")
        ax.grid(False)
    plt.tight_layout(); return fig


key_models = [BEST_LINEAR] + (["M9 DistilBERT", "M10 Ensemble"] if CFG.run_bert else [])
fig = plot_cm_grid(key_models, "natural"); savefig(fig, "natural_confusion"); plt.show()
nat_counts = nat_df["label"].value_counts()[LABELS]
print("Natural test set composition:", ", ".join(f"{l} {n:,} ({n / len(nat_df):.1%})" for l, n in nat_counts.items()))

TODO-INTERP: natural confusion

### Deployment recommendation

TODO-INTERP: recommendation

<a id="s8"></a>
## 8. Business insights

The models are a means to an end. This section turns their predictions into things a product team can act on, by **combining the text predictions with stars, photos, helpful votes, products, stores and prices**.

Two scorers are used:
* **The best model** (highest *validation* macro F1) for detailed analysis of the 10,000-review natural test set, which no model trained on.
* **The best linear model** to score the **whole corpus** cheaply (every review not used for training or validation). This is the kind of high-volume job the leaderboard suggested a linear model for.

In [ ]:
val_rank = BOOK.table("val")
BEST_MODEL = val_rank.iloc[0]["model"]
best_nat = pd.read_parquet(CFG.dirs["predictions"] / f"{''.join(c if c.isalnum() else '_' for c in BEST_MODEL)}__natural.parquet")
nat = nat_df.merge(best_nat[["review_id", "y_pred", "p_negative", "p_neutral", "p_positive"]], on="review_id")
nat["pred_label"] = nat["y_pred"].map(dict(enumerate(LABELS)))
print(f"Best model by validation macro F1: {BEST_MODEL} ({val_rank.iloc[0]['macro_f1']:.3f})")

# Score the whole corpus (minus train/val) with the best linear model
t0 = time.time()
scorer = LINEAR_CANDIDATES[BEST_LINEAR]
corpus = df[~df["review_id"].isin(set(train_df["review_id"]) | set(val_df["review_id"]))].reset_index(drop=True)
cols = ["text_classical", "text_light"] if "M2" in BEST_LINEAR else "text_classical"
chunks = [scorer.predict_proba(corpus.iloc[i:i + 50_000][cols]) for i in range(0, len(corpus), 50_000)]
corpus[["p_negative", "p_neutral", "p_positive"]] = np.vstack(chunks)
corpus["pred_label"] = np.array(LABELS)[corpus[["p_negative", "p_neutral", "p_positive"]].values.argmax(1)]
corpus[["review_id", "p_negative", "p_neutral", "p_positive", "pred_label"]].to_parquet(
    CFG.dirs["predictions"] / "corpus_scores_best_linear.parquet", index=False)
print(f"Scored {len(corpus):,} corpus reviews with {BEST_LINEAR} in {time.time() - t0:.0f}s "
      f"({(time.time() - t0) / len(corpus) * 1e6 / 60:.1f} min per million reviews)")
agree = (corpus["pred_label"] == corpus["label"]).mean()
print(f"Agreement between predicted sentiment and star-based label on the corpus: {agree:.1%}")

### 8a. When the stars and the words disagree
**Hidden complaints** are 4 to 5-star reviews whose text the best model reads as negative. **Hidden praise** is the reverse: 1 to 2-star reviews that read positive. Both matter:
* hidden complaints are problems that star-based dashboards never see (the customer liked the product overall but reports a defect);
* hidden praise is often a complaint about something *other than* the product (delivery, a wrong item), or a customer who picked the wrong star. These should go to a different team.

In [ ]:
pos_stars, neg_stars = nat["label"] == "positive", nat["label"] == "negative"
hidden_complaints = nat[pos_stars & (nat["pred_label"] == "negative")].sort_values("p_negative", ascending=False)
hidden_praise = nat[neg_stars & (nat["pred_label"] == "positive")].sort_values("p_positive", ascending=False)
mismatch = pd.DataFrame({
    "reviews": [pos_stars.sum(), neg_stars.sum()],
    "text reads opposite": [len(hidden_complaints), len(hidden_praise)],
    "...with confidence >= 80%": [(hidden_complaints["p_negative"] >= 0.8).sum(), (hidden_praise["p_positive"] >= 0.8).sum()],
}, index=["4-5 star reviews (hidden complaints)", "1-2 star reviews (hidden praise)"])
mismatch["share_%"] = (mismatch["text reads opposite"] / mismatch["reviews"] * 100).round(2)
display(mismatch)

# The same check at corpus scale with the linear scorer (confidence >= 80% to keep it conservative)
c_hc = ((corpus["label"] == "positive") & (corpus["p_negative"] >= 0.8)).sum()
print(f"Corpus scale ({BEST_LINEAR}, P(negative) >= 0.8): {c_hc:,} of {(corpus['label'] == 'positive').sum():,} "
      f"4-5 star reviews ({c_hc / (corpus['label'] == 'positive').sum():.2%}) read as clearly negative.")

show = lambda d, pcol: display(d.head(6)[["rating", pcol, "review"]].style.format({pcol: "{:.2f}"}).hide(axis="index")
                                .set_properties(subset=["review"], **{"text-align": "left", "white-space": "normal"}))
display(Markdown("**Most confident hidden complaints (4-5 stars, text reads negative):**")); show(hidden_complaints, "p_negative")
display(Markdown("**Most confident hidden praise (1-2 stars, text reads positive):**")); show(hidden_praise, "p_positive")

TODO-INTERP: mismatch

### 8b. Aspect analysis: what are customers talking about?
A small keyword lexicon tags each review with the product aspects it mentions. A lexicon is fully transparent (a product manager can read and edit it) and runs in seconds on the whole corpus. For each aspect we measure:
* **mention rate**: share of all reviews that mention it;
* **negative share when mentioned**: share of those reviews that are 1 to 2 stars, and the **lift** over the average review;
* **coverage of negative reviews**: share of all negative reviews that mention it.

An aspect with both high lift and high coverage is a **driver of negative reviews**.

In [ ]:
ASPECTS = {
    "scent": r"\b(?:smell\w*|scent\w*|fragrance\w*|odou?r\w*|perfume\w*|stink\w*)",
    "packaging / leakage": r"\b(?:leak\w*|spill\w*|packag\w*|bottle\w*|pump\w*|lid|lids|cap|caps|seal\w*|cracked|broken|container|tube|dispenser)\b",
    "price / value": r"\b(?:price\w*|expensive|overpriced|pricey|cheap\w*|value|worth|money|cost\w*)\b",
    "skin reaction": r"\b(?:rash\w*|irritat\w*|burn\w*|itch\w*|breakout\w*|broke me out|broken out|acne|pimple\w*|allerg\w*|redness|sting\w*|hives|swell\w*|react\w*)",
    "texture": r"\b(?:texture\w*|greasy|oily|sticky|thick|thin|runny|watery|creamy|consistency|clump\w*|gritty|chalky|lather\w*|absorb\w*)\b",
    "longevity": r"\b(?:lasts?|lasting|long[- ]lasting|wears? off|wore off|fade\w*|all day|stays? on|staying power)\b",
    "shade / colour": r"\b(?:shades?|colou?rs?|tones?|pigment\w*|darker|lighter|match\w*)\b",
    "shipping / delivery": r"\b(?:ship\w*|deliver\w*|arriv\w*|seller|courier|tracking)\b",
    "authenticity / fake": r"\b(?:fake|counterfeit|knock[- ]?offs?|authentic\w*|genuine|expired|expiration|tampered|not the real)\b",
}
SAFETY_ASPECTS = ["skin reaction", "authenticity / fake"]

t0 = time.time()
lower = df["review"].str.lower()
for a, pat in ASPECTS.items():
    df[f"asp_{a}"] = lower.str.contains(pat, regex=True)
asp_cols = [f"asp_{a}" for a in ASPECTS]
for frame in (corpus, nat):
    frame[asp_cols] = df.set_index("review_id").loc[frame["review_id"], asp_cols].values
print(f"Tagged {len(df):,} reviews with {len(ASPECTS)} aspects in {time.time() - t0:.0f}s; "
      f"{df[asp_cols].any(axis=1).mean():.1%} mention at least one aspect")

base_neg = (df["label"] == "negative").mean()
rows = []
for a in ASPECTS:
    m = df[f"asp_{a}"]
    sub = df.loc[m, "label"].value_counts(normalize=True)
    rows.append({"aspect": a, "mention_rate_%": m.mean() * 100,
                 "negative_%": sub.get("negative", 0) * 100, "neutral_%": sub.get("neutral", 0) * 100,
                 "positive_%": sub.get("positive", 0) * 100,
                 "lift_vs_avg": sub.get("negative", 0) / base_neg,
                 "coverage_of_negatives_%": m[df["label"] == "negative"].mean() * 100,
                 "predicted_negative_%": (corpus.loc[corpus[f"asp_{a}"], "pred_label"] == "negative").mean() * 100})
aspect_table = pd.DataFrame(rows).set_index("aspect").sort_values("negative_%", ascending=False)
display(aspect_table.round(1))
aspect_table.to_csv(CFG.dirs["results"] / "aspect_table.csv")

fig, axes = plt.subplots(1, 2, figsize=(17, 4.8), gridspec_kw={"width_ratios": [1.1, 1]})
stacked_share_bar(axes[0], aspect_table[["negative_%", "neutral_%", "positive_%"]].set_axis(LABELS, axis=1).iloc[::-1],
                  "Sentiment of reviews that mention each aspect (star labels)")
axes[0].axvline(base_neg * 100, color=INK, ls="--", lw=1)
axes[0].text(base_neg * 100 + 0.5, len(aspect_table) - 0.45, f"average review: {base_neg:.0%} negative", fontsize=8, color=INK)
ax = axes[1]
ax.scatter(aspect_table["coverage_of_negatives_%"], aspect_table["lift_vs_avg"], s=aspect_table["mention_rate_%"] * 40,
           color=SENT_COLORS["negative"], alpha=0.75, edgecolor="white", linewidth=1.5)
for a, r in aspect_table.iterrows():
    ax.annotate(a, (r["coverage_of_negatives_%"], r["lift_vs_avg"]), xytext=(7, 4), textcoords="offset points", fontsize=9)
ax.axhline(1, color=INK2, ls="--", lw=1)
ax.set(xlabel="Share of ALL negative reviews that mention the aspect (%)", ylabel="Negative share vs average review (lift)",
       title="Drivers of negative reviews (bubble size = mention rate)")
plt.tight_layout(); savefig(fig, "aspects"); plt.show()

hc_aspects = pd.DataFrame({"hidden complaints": nat.loc[hidden_complaints.index, asp_cols].mean() * 100,
                           "all 4-5 star reviews": nat.loc[pos_stars, asp_cols].mean() * 100})
hc_aspects.index = list(ASPECTS)
hc_aspects["ratio"] = hc_aspects["hidden complaints"] / hc_aspects["all 4-5 star reviews"]
display(Markdown("**What are the hidden complaints (8a) about?** Aspect mention rates (%), hidden complaints vs all 4-5 star reviews:"))
display(hc_aspects.sort_values("ratio", ascending=False).round(2))

TODO-INTERP: aspects

### 8c. Product and store watchlist
Products and stores with enough reviews to judge (at least 100 reviews for a product, 300 for a store), ranked by the **predicted negative share** from the corpus scorer. The **gap** column is predicted negative share minus star-based negative share: a positive gap means the text is more negative than the stars, so problems are being under-reported. The **top aspect** is the aspect mentioned most often in that product's negative-predicted reviews, which tells the team where to look first.

In [ ]:
def top_aspect(frame):
    neg = frame[frame["pred_label"] == "negative"]
    if len(neg) == 0:
        return "-"
    rates = neg[asp_cols].mean()
    return f"{rates.idxmax().replace('asp_', '')} ({rates.max():.0%})"


corpus["pred_negative"] = corpus["pred_label"] == "negative"
corpus["star_negative"] = corpus["label"] == "negative"


def watchlist(group_col, min_reviews, top=12):
    g = corpus.groupby(group_col)
    t = pd.DataFrame({"reviews": g.size(), "avg_stars": g["rating"].mean(),
                      "pred_negative_%": g["pred_negative"].mean() * 100, "star_negative_%": g["star_negative"].mean() * 100})
    t = t[t["reviews"] >= min_reviews]
    t["gap_pts"] = t["pred_negative_%"] - t["star_negative_%"]
    t = t.sort_values("pred_negative_%", ascending=False).head(top)
    t["top_aspect_in_negatives"] = [top_aspect(corpus[corpus[group_col] == k]) for k in t.index]
    return t, (g.size() >= min_reviews).sum()


prod_watch, n_prod = watchlist("parent_asin", 100)
titles = corpus.drop_duplicates("parent_asin").set_index("parent_asin")[["product_title", "store"]]
prod_watch = prod_watch.join(titles)
prod_watch["product_title"] = prod_watch["product_title"].fillna("(no title)").str.slice(0, 55)
display(Markdown(f"**Product watchlist** ({n_prod:,} products have at least 100 scored reviews):"))
display(prod_watch[["product_title", "store", "reviews", "avg_stars", "pred_negative_%", "star_negative_%", "gap_pts",
                    "top_aspect_in_negatives"]].round(1))

store_watch, n_store = watchlist("store", 300, top=10)
display(Markdown(f"**Store watchlist** ({n_store:,} stores have at least 300 scored reviews):"))
display(store_watch.round(1))
prod_watch.to_csv(CFG.dirs["results"] / "watchlist_products.csv"); store_watch.to_csv(CFG.dirs["results"] / "watchlist_stores.csv")

In [ ]:
band_order = PRICE_LABELS + ["Unknown"]
pb = corpus.groupby("price_band").agg(reviews=("review_id", "size"),
                                      pred_negative=("pred_label", lambda s: (s == "negative").mean() * 100),
                                      star_negative=("label", lambda s: (s == "negative").mean() * 100),
                                      pred_neutral=("pred_label", lambda s: (s == "neutral").mean() * 100)).reindex(band_order)
fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(pb))
ax.bar(x - 0.2, pb["star_negative"], width=0.38, color="#f2a3a2", label="negative share (stars)")
ax.bar(x + 0.2, pb["pred_negative"], width=0.38, color=SENT_COLORS["negative"], label="negative share (predicted from text)")
for xi, v in zip(x + 0.2, pb["pred_negative"]):
    ax.text(xi, v + 0.3, f"{v:.1f}%", ha="center", fontsize=9)
ax.set(xticks=x, xticklabels=[f"{b}\n(n={n:,})" for b, n in zip(pb.index, pb["reviews"])], ylabel="Share of reviews",
       title="Negative sentiment by product price band")
ax.yaxis.set_major_formatter(mtick.PercentFormatter()); ax.legend(); ax.grid(axis="x", visible=False)
plt.tight_layout(); savefig(fig, "price_band"); plt.show()
display(pb.round(1))

TODO-INTERP: watchlist and price

### 8d. Photos: what do customer images add?
The brief mentions that reviews include customer photos. Before using any image model, the simplest question is what the *presence* of a photo tells us, and what photo reviews talk about.

In [ ]:
photo_tbl = df.groupby("has_image").agg(reviews=("review_id", "size"), negative_pct=("label", lambda s: (s == "negative").mean() * 100),
                                        median_words=("word_count", "median"), mean_helpful=("helpful_vote", "mean"))
photo_tbl.index = ["no photo", "with photo"]
display(photo_tbl.round(2))

neg = df[df["label"] == "negative"]
photo_aspects = pd.DataFrame({"negative, with photo": neg.loc[neg["has_image"], asp_cols].mean() * 100,
                              "negative, no photo": neg.loc[~neg["has_image"], asp_cols].mean() * 100})
photo_aspects.index = list(ASPECTS)
photo_aspects["ratio"] = photo_aspects["negative, with photo"] / photo_aspects["negative, no photo"]
photo_aspects = photo_aspects.sort_values("ratio", ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
yy = np.arange(len(photo_aspects))
ax.barh(yy + 0.2, photo_aspects["negative, with photo"], height=0.38, color=SENT_COLORS["negative"], label="negative reviews WITH a photo")
ax.barh(yy - 0.2, photo_aspects["negative, no photo"], height=0.38, color="#f2a3a2", label="negative reviews without a photo")
ax.set(yticks=yy, yticklabels=photo_aspects.index, xlabel="Share of negative reviews mentioning the aspect (%)",
       title="What negative reviews with photos are about")
ax.invert_yaxis(); ax.legend(loc="lower right"); ax.grid(axis="y", visible=False)
plt.tight_layout(); savefig(fig, "photo_aspects"); plt.show()
display(photo_aspects.round(1))

acc_by_photo = nat.groupby("has_image").apply(lambda g: pd.Series({"reviews": len(g), "accuracy": (g["y"] == g["y_pred"]).mean(),
                                                                   "macro_f1": f1_score(g["y"], g["y_pred"], average="macro")}))
acc_by_photo.index = ["no photo", "with photo"]
print(f"{BEST_MODEL} on the natural test set, by photo presence:")
display(acc_by_photo.round(3))

TODO-INTERP: photos

**Optional pilot: CLIP image embeddings (flagged by `CFG.run_clip_pilot`).** CLIP (OpenAI, 2021) places images and short texts in the same space, so it can score a photo against prompts such as "a photo of a leaking bottle" with no training (zero-shot). The pilot downloads a small sample of review photos, scores each against "problem" and "no problem" prompts, and checks whether the image score agrees with the text model's P(negative). It is written to **skip cleanly** if the CLIP model or the photos cannot be downloaded (for example, if the image host blocks the request or the URL has expired).

In [ ]:
CLIP_PROMPTS = {"problem": ["a photo of a broken or leaking product", "a photo of a damaged package",
                            "a photo of a skin rash or irritation", "a photo of a product that looks fake or used"],
                "fine": ["a photo of a beauty product in good condition", "a photo of makeup applied nicely",
                         "a photo of a happy customer using a product", "a photo of healthy hair or skin"]}


def run_clip_pilot(sample_n=150, timeout=6):
    import io, requests
    from PIL import Image
    from transformers import CLIPModel, CLIPProcessor
    try:
        clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
        proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    except Exception as e:
        print(f"CLIP pilot skipped: could not load the CLIP model ({type(e).__name__}: {str(e)[:120]})")
        return None
    pool_ = nat[nat["first_image_url"].notna()]
    pool_ = pd.concat([pool_[pool_["label"] == l].head(sample_n // 3) for l in LABELS])
    images, keep = [], []
    for i, url in pool_["first_image_url"].items():
        try:
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            images.append(Image.open(io.BytesIO(r.content)).convert("RGB")); keep.append(i)
        except Exception:
            continue
    print(f"Downloaded {len(images)} of {len(pool_)} review photos")
    if len(images) < 20:
        print("CLIP pilot skipped: too few photos could be downloaded.")
        return None
    texts = CLIP_PROMPTS["problem"] + CLIP_PROMPTS["fine"]
    with torch.no_grad():
        out = clip(**proc(text=texts, images=images, return_tensors="pt", padding=True))
    probs = out.logits_per_image.softmax(-1).numpy()
    res = pool_.loc[keep, ["review_id", "label", "p_negative", "review"]].copy()
    res["image_problem_score"] = probs[:, :len(CLIP_PROMPTS["problem"])].sum(1)
    print(f"Correlation between image problem score and text P(negative): {res['image_problem_score'].corr(res['p_negative']):.3f}")
    display(res.groupby("label")["image_problem_score"].mean().loc[LABELS].round(3).to_frame("mean image problem score"))
    res.to_csv(CFG.dirs["results"] / "clip_pilot.csv", index=False)
    return res


clip_res = run_clip_pilot() if CFG.run_clip_pilot else print("CLIP pilot disabled in CONFIG.")

TODO-INTERP: clip

### 8e. Helpfulness: do shoppers find negative reviews more helpful?
Longer reviews collect more votes, and negative reviews are longer (Section 3.2), so votes are compared **within the same review-length band**. Separately, we look at which sentiments make up the 1% most-helpful reviews.

In [ ]:
len_bins = pd.cut(df["word_count"], [0, 20, 50, 100, 200, np.inf], labels=["1-20", "21-50", "51-100", "101-200", "200+"])
helpful_by_len = df.groupby([len_bins, "label"], observed=True)["helpful_vote"].mean().unstack()[LABELS]
cut = df["helpful_vote"].quantile(0.99)
top1 = df[df["helpful_vote"] >= cut]
share = pd.DataFrame({"share of all reviews %": df["label"].value_counts(normalize=True)[LABELS] * 100,
                      "share of top-1% most helpful %": top1["label"].value_counts(normalize=True)[LABELS] * 100})

fig, axes = plt.subplots(1, 2, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1.4, 1]})
xb = np.arange(len(helpful_by_len))
for off, lab in zip([-0.27, 0, 0.27], LABELS):
    axes[0].bar(xb + off, helpful_by_len[lab], width=0.26, color=SENT_COLORS[lab], label=lab)
axes[0].set(xticks=xb, xticklabels=helpful_by_len.index, xlabel="Review length (words)", ylabel="Average helpful votes",
            title="Helpful votes by sentiment, within each length band"); axes[0].legend(); axes[0].grid(axis="x", visible=False)
stacked_share_bar(axes[1], share.T.rename(index={"share of all reviews %": "all reviews",
                                                 "share of top-1% most helpful %": f"top 1% most helpful\n(≥{cut:.0f} votes)"}),
                  "Who writes the most helpful reviews?")
plt.tight_layout(); savefig(fig, "helpfulness"); plt.show()
display(helpful_by_len.round(2)); display(share.round(1))

TODO-INTERP: helpful

### 8f. Putting it together: a complaint triage queue
A product team cannot read every negative review either. The last step combines the signals above into one transparent **priority score** for each review in the natural test set:

> priority = P(negative) × (1 + log(1 + helpful votes)) × (1.5 if it has a photo) × (2 if it mentions a skin reaction or a fake product)

Each part has a business reason: P(negative) from the text model, helpful votes for how many shoppers it influences, photos as evidence, and safety or authenticity issues for legal and health risk. The multipliers are a **starting point for the business to tune**, not learned values.

In [ ]:
q = nat.copy()
q["safety_flag"] = q[[f"asp_{a}" for a in SAFETY_ASPECTS]].any(axis=1)
q["priority"] = (q["p_negative"] * (1 + np.log1p(q["helpful_vote"])) * np.where(q["has_image"], 1.5, 1.0)
                 * np.where(q["safety_flag"], 2.0, 1.0))
q["aspects"] = q[asp_cols].apply(lambda r: ", ".join(a for a, v in zip(ASPECTS, r) if v) or "-", axis=1)
queue = q.sort_values("priority", ascending=False).head(10)
display(queue[["priority", "rating", "p_negative", "helpful_vote", "has_image", "aspects", "review"]]
        .assign(review=lambda d: d["review"].str.slice(0, 220))
        .style.format({"priority": "{:.2f}", "p_negative": "{:.2f}"}).hide(axis="index")
        .set_properties(subset=["review"], **{"text-align": "left", "white-space": "normal"}))
top100 = q.sort_values("priority", ascending=False).head(100)
print(f"Top 100 of the queue: {top100['safety_flag'].mean():.0%} safety/authenticity, {top100['has_image'].mean():.0%} with photos, "
      f"{(top100['label'] != 'negative').mean():.0%} were NOT 1-2 star reviews (found only through the text).")
queue.to_csv(CFG.dirs["results"] / "triage_queue_top10.csv", index=False)

TODO-INTERP: triage

<a id="s9"></a>
## 9. Reusable inference

**Why:** a model is only useful if other systems can call it. The `SentimentPredictor` class below:
* loads the saved model from `outputs/models/` **once** (the ensemble if the DistilBERT folder is present, otherwise the best linear model), so it works in a fresh session without retraining anything;
* applies **exactly the same cleaning** as in training (it calls the Section 2 functions);
* returns, for each text, the label, the confidence, all three probabilities and a `needs_human_review` flag for low-confidence cases.

`predict_sentiment(texts)` is a one-line shortcut for the common case.

In [ ]:
class SentimentPredictor:
    """Load the saved model once, then classify any number of new texts.

    backend: "auto" (ensemble if DistilBERT is saved, else linear), "ensemble", "bert" or "linear".
    """

    def __init__(self, model_dir=CFG.dirs["models"], backend="auto", device=None, low_confidence=0.6):
        self.dir = Path(model_dir)
        self.cfg = load_json(self.dir / "deployment.json")
        self.low_confidence = low_confidence
        self.linear = joblib.load(self.dir / self.cfg["linear_file"])
        bert_dir = self.dir / self.cfg["bert_dir"]
        use_bert = backend in ("auto", "ensemble", "bert") and (bert_dir / "config.json").exists()
        if backend in ("ensemble", "bert") and not use_bert:
            raise FileNotFoundError(f"DistilBERT weights not found in {bert_dir}")
        self.backend = ("bert" if backend == "bert" else "ensemble") if use_bert else "linear"
        if use_bert:
            self.device = device or detect_device()
            self.tokenizer = AutoTokenizer.from_pretrained(bert_dir)
            self.bert = AutoModelForSequenceClassification.from_pretrained(bert_dir, dtype=torch.float32).to(self.device).eval()

    def _linear_proba(self, texts):
        frame = pd.DataFrame({"text_light": [clean_light(t) for t in texts],
                              "text_classical": [clean_classical(t) or "emptyreview" for t in texts]})
        return self.linear.predict_proba(frame[self.cfg["linear_columns"]])

    @torch.no_grad()
    def _bert_proba(self, texts, batch_size=32):
        out = []
        for i in range(0, len(texts), batch_size):
            enc = self.tokenizer([clean_light(t) for t in texts[i:i + batch_size]], truncation=True,
                                 max_length=self.cfg["bert_max_len"], padding=True, return_tensors="pt").to(self.device)
            out.append(torch.softmax(self.bert(**enc).logits.float(), -1).cpu().numpy())
        return np.vstack(out)

    def predict_proba(self, texts) -> np.ndarray:
        texts = [combine_title_text("", t) for t in texts]       # same HTML clean-up as training
        if self.backend == "linear":
            return self._linear_proba(texts)
        if self.backend == "bert":
            return self._bert_proba(texts)
        w = self.cfg["w_bert"]
        return w * self._bert_proba(texts) + (1 - w) * self._linear_proba(texts)

    def predict(self, texts) -> pd.DataFrame:
        texts = [texts] if isinstance(texts, str) else list(texts)
        p = self.predict_proba(texts)
        res = pd.DataFrame(p, columns=[f"p_{l}" for l in LABELS])
        res.insert(0, "label", np.array(LABELS)[p.argmax(1)])
        res.insert(1, "confidence", p.max(1))
        res["needs_human_review"] = res["confidence"] < self.low_confidence
        res.insert(0, "text", texts)
        return res


_PREDICTOR = None


def predict_sentiment(texts, backend="auto") -> pd.DataFrame:
    """Shortcut: classify one text or a list of texts with the saved deployed model."""
    global _PREDICTOR
    if _PREDICTOR is None or _PREDICTOR.backend != backend and backend != "auto":
        _PREDICTOR = SentimentPredictor(backend=backend)
    return _PREDICTOR.predict(texts)

In [ ]:
linear_files = {"M1 Count+chi2+LR": "M1_Count_chi2_LR.joblib", "M2 TF-IDF word+char LR": "M2_TFIDF_word_char_LR.joblib",
                "M3 TF-IDF+ComplementNB": "M3_TFIDF_ComplementNB.joblib", "M4 TF-IDF+LinearSVC": "M4_TFIDF_LinearSVC.joblib"}
deployment = {"linear_model": BEST_LINEAR, "linear_file": linear_files[BEST_LINEAR],
              "linear_columns": ["text_classical", "text_light"] if "M2" in BEST_LINEAR else "text_classical",
              "bert_dir": "M9_DistilBERT", "bert_max_len": CFG.bert_max_len,
              "w_bert": W_BERT if CFG.run_bert else 0.0, "labels": LABELS}
save_json(deployment, CFG.dirs["models"] / "deployment.json")
display(pd.Series(deployment, name="value").to_frame())

**Demo on new reviews** (written for this demo, not taken from the dataset). They include the hard cases: negation, mixed feelings, a hidden complaint, sarcasm, and two texts that are not reviews at all (a chat message and a social-media post). The demo compares the deployed model with the linear-only fallback.

In [ ]:
new_reviews = [
    "Absolutely love this serum, my skin has never looked better!",
    "Not bad at all, actually better than I expected for the price.",
    "It's okay. Does what it says but nothing special, probably won't repurchase.",
    "Five stars for the colour, but the pump broke after a week and it leaked all over my bag.",
    "Gave me a burning rash within an hour. Threw it away.",
    "This is clearly a fake, the packaging is different from the one I bought in the store.",
    "Oh great, another 'moisturiser' that leaves my face feeling like sandpaper. Just what I needed.",
    "Hi, my order arrived yesterday but the lid was cracked. Can I get a replacement?",       # customer-service chat
    "obsessed w this lip oil sm, def my new everyday fave",                                  # social-media style
]
t0 = time.time()
deployed = SentimentPredictor()
print(f"Loaded backend '{deployed.backend}' in {time.time() - t0:.1f}s")
t0 = time.time(); res = deployed.predict(new_reviews); t_dep = time.time() - t0
lin_only = SentimentPredictor(backend="linear").predict(new_reviews)
res["linear_only_label"] = lin_only["label"].values
display(res.round(3).style.format(precision=2).hide(axis="index")
        .set_properties(subset=["text"], **{"text-align": "left", "white-space": "normal"}))
print(f"Predicted {len(new_reviews)} texts in {t_dep:.2f}s with the '{deployed.backend}' backend")

TODO-INTERP: demo

In [ ]:
# The one-line shortcut, e.g. for a customer-service message that has no star rating
predict_sentiment("Smells like cheap plastic and it dried out my hair.").round(3)

<a id="s10"></a>
## 10. Conclusions, limitations, ethics and future work

TODO-INTERP: conclusions

In [ ]:
total_min = (time.time() - NOTEBOOK_T0) / 60
summary = {"notebook_run_minutes_this_run": round(total_min, 1),
           "deep_models_loaded_from_checkpoint": not CFG.force_retrain_deep,
           "original_training_minutes": {r["model"]: round(r["train_time_s"] / 60, 1)
                                         for _, r in BOOK.table("test").iterrows()}}
save_json(summary, CFG.dirs["results"] / "run_summary.json")
print(f"This run of the notebook took {total_min:.1f} minutes on {HARDWARE['CPU']} ({CFG.device}).")
print("Original training time per model (minutes):")
display(pd.Series(summary["original_training_minutes"], name="train minutes").to_frame())

<a id="notes"></a>
## Notes for Dylan

TODO-INTERP: notes